# Phase 2 Triage and Routing Preparation

I test the existing official triage using explicitly synthetic inputs. I do not load Validation Predictions or the final Test Set. No percentage here is an official Validation Assignment Rate.

Contents: setup, input format, loading guard, adapter, batch lifecycle, verification, metrics, synthetic tests, pending dependencies.

## 1. Project and contract setup
I resolve the project from SVR_PROJECT_ROOT or the current directory and its parents. I set SVR_ARAFA_ROOT for the separate clean official checkout; otherwise I use the sibling svr-arafa-review convention. I verify the pinned SHA and source blobs before imports. I run from the project root or notebook directory in a fresh isolated Python process, never an API worker. No checkout is changed or downloaded.

In [1]:
import copy, csv, hashlib, io, itertools, json, os, platform, stat, subprocess, sys, unittest
from contextlib import ExitStack
from collections import Counter
from pathlib import Path
from unittest.mock import patch
sys.dont_write_bytecode = True
SHA = "fe695befbb9bd8769273cc7ea2e8dffc84518820"

def resolve_project_root():
    configured = os.environ.get("SVR_PROJECT_ROOT")
    start = Path(configured).expanduser().resolve() if configured else Path.cwd().resolve()
    candidates = [start] if configured else [start, *start.parents]
    for candidate in candidates:
        if ((candidate / ".git").exists()
                and (candidate / "notebooks/triage-routing/triage_routing_phase2.ipynb").is_file()
                and (candidate / "docs/project/roles-and-workflow.md").is_file()):
            return candidate
    raise RuntimeError("Project root not found. Run from the project root or notebook directory, "
                       "or set SVR_PROJECT_ROOT to the checkout containing this notebook.")

ROOT = resolve_project_root()
REVIEW = Path(os.environ.get("SVR_ARAFA_ROOT", str(ROOT.parent / "svr-arafa-review"))).expanduser().resolve()
if not (REVIEW / ".git").exists():
    raise RuntimeError("Official source checkout missing. Set SVR_ARAFA_ROOT to an existing Git "
                       f"checkout pinned to {SHA}. No source is downloaded automatically.")

def git(root, *args):
    try:
        return subprocess.check_output(
            ["git", "--no-optional-locks", "-c", f"safe.directory={root.as_posix()}", "-C", str(root), *args],
            text=True, stderr=subprocess.PIPE).strip()
    except FileNotFoundError as exc:
        raise RuntimeError("Git is required on PATH. Install/configure it before running setup.") from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"Read-only Git verification failed: {exc.stderr.strip()}") from exc

actual_commit = git(REVIEW, "rev-parse", "HEAD")
if actual_commit != SHA:
    raise RuntimeError(f"Official commit mismatch: expected {SHA}, found {actual_commit}. "
                       "Point SVR_ARAFA_ROOT at the reviewed checkout; do not update the pin without review.")
if git(REVIEW, "status", "--porcelain"):
    raise RuntimeError("Official checkout has local changes. Use a clean reviewed checkout; "
                       "this notebook will not alter or clean it.")
if any(name == "svr" or name.startswith("svr.") or name == "infra" or name.startswith("infra.")
       for name in sys.modules):
    raise RuntimeError("Official modules are already loaded. Restart the kernel or use a fresh Python process.")

SOURCE_HASHES = {}
source_paths = git(REVIEW, "ls-tree", "-r", "--name-only", SHA, "--", "src/svr", "infra").splitlines()
for relative in source_paths:
    if not relative.endswith(".py"):
        continue
    path = REVIEW / relative
    blob = subprocess.check_output(["git", "-c", f"safe.directory={REVIEW.as_posix()}",
                                   "-C", str(REVIEW), "show", f"{SHA}:{relative}"])
    if not path.is_file() or path.read_bytes().replace(b"\r\n", b"\n") != blob.replace(b"\r\n", b"\n"):
        raise RuntimeError(f"Official source differs from pinned commit: {relative}. Use a clean reviewed checkout.")
    SOURCE_HASHES[relative] = hashlib.sha256(blob).hexdigest()
for relative in ("src/svr/schemas.py", "src/svr/agents.py", "src/svr/stages/triage.py",
                 "src/svr/stages/ticket.py", "infra/predictions_file.py"):
    if relative not in SOURCE_HASHES:
        raise RuntimeError(f"Required official module missing: {relative}. Check SVR_ARAFA_ROOT.")

sys.path[:0] = [str(REVIEW / "src"), str(REVIEW)]
from svr import agents
from svr.schemas import TextAnalysis, VoiceEvidence, ComplaintCategory, SCHEMA_VERSION, LABEL_VERSION
from svr.stages.triage import triage_phase1
from svr.stages.ticket import assemble_ticket
from tempfile import TemporaryDirectory
from infra.predictions_file import PredictionsFileError, SCHEMA_VERSION as PREDICTIONS_SCHEMA_VERSION, read_predictions_file
from svr.schemas import ChurnRisk, ComplaintUrgency, CustomerState
import pydantic
if Path(sys.modules["svr"].__file__).resolve() != REVIEW / "src/svr/__init__.py":
    raise RuntimeError("Unexpected svr import origin. Restart in a fresh isolated environment.")
assert SCHEMA_VERSION == 1 and LABEL_VERSION == "taxonomy-2026-11-15"
def snapshot():
    return [a.model_dump(mode="json") for a in agents.list_agents()]
BASELINE = snapshot()
print(json.dumps({"commit": SHA, "schema_version": SCHEMA_VERSION, "label_version": LABEL_VERSION,
                  "python": platform.python_version(), "pydantic": pydantic.__version__,
                  "source_sha256": SOURCE_HASHES}, indent=2))

{
  "commit": "fe695befbb9bd8769273cc7ea2e8dffc84518820",
  "schema_version": 1,
  "label_version": "taxonomy-2026-11-15",
  "python": "3.12.10",
  "pydantic": "2.14.3",
  "source_sha256": {
    "infra/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "infra/experiment_config.py": "33a616f4aeea807ef259499b54784187ee6a25243d734b0012a80d0bbd2a213d",
    "infra/harness.py": "ef5b8a95ce6c1270c058623b75914468431b806256529b544e565166e80580ce",
    "infra/metrics.py": "55d442eddea1758bed54e7e83d18febdb701c980f6ef9574d03aaec0ce4b826c",
    "infra/mock_predictions.py": "79a39621b7119ba00b5bb3d7a6868cf45433c04dd0fbb48831f9f4a0b068e08a",
    "infra/predictions_file.py": "450209140a8ba27a64b716b0535a7268e0ebb54574927b304b6496d9fae62cf0",
    "infra/run_record.py": "2e601d70f057eca107df556ec52dcbbba7ab8cee38efbd05e9f4360a11732430",
    "infra/seeds.py": "301724b9794e6bc1c714fc13dba3a5656e2d729bbbec9fb9bdf5378c86753911",
    "infra/shared_config.py": "53a02f3256b3

## 2. Input format inspection
The official infra reader returns harness_rows, probabilities and metadata. Its customer_state maps explicitly to customer_state_text. The wire split is val and maps to validation internally. I pass the four approved export customer_state classes to the official reader. The runtime enum still accepts unclear. I retain the exported churn argmax label. Prediction-file schema_version is separate from runtime schema_version.

## 3. Validation Predictions loading
Hazem confirmed fair-run MARBERTv2 from runs/exp_fair-marbertv2-lr2e5 and the val manifest from the ready-branch clean_training_val.csv. Run All exercises synthetic fixtures only. The separate Real Validation entry point requires explicit authorization and authenticates the six allowlisted artifacts before routing. I exercise the loader with temporary synthetic fixtures.

## 4. Contract validation
I populate TextAnalysis with the pinned runtime versions and derive voicemail_id directly from audio_file. The Validation reader handoff needs no per-row supplements. Valid null labels require Human Review. Invalid enums, unsupported fields and malformed rows remain errors.

## 5. Prediction-to-TextAnalysis adapter
I join predictions by audio_file to the text column of final/clean_training_val.csv. The full corrected text goes to the ticket transcript. The provisional summary is the first 200 Python characters. The provisional customer_request describes the predicted category without verifying the customer's wording. Missing, blank or non-string text is a row error. A null category leaves customer_request empty and triggers review. Unsupported categories are row errors.

For complete rows, confidence is the minimum predicted-class probability across the four tasks. Below-threshold confidence or an explicit null runtime label triggers pre-routing review. A valid Unassigned outcome also triggers final ticket review. The proposed default threshold is 0.5. A null-label row requires explicit confidence. I populate suggested_script for valid categories from PROPOSED_SUGGESTED_SCRIPTS. Hazem's approval is pending for customer-facing use. A null category requires explicit upstream script text.

## 6. Batch Triage execution
I process tickets sequentially under Hazem's confirmed policy. Official triage reserves capacity, and I release each returned reservation once in finally before the next ticket. The runner owns that release and does not call TicketStore.complete. Both the synthetic and authorized runners report sequential-validation metrics.

## 7. Priority verification
An independent oracle checks the approved points and exact buckets; it never creates routing decisions.

## 8. Routing eligibility verification
I verify the first eligible Agent in the pre-reservation Seed Order using matching skill, shift and capacity. Tier and lowest-load ranking are ignored.

## 9. Human Review and Missing Inputs
Null scoring inputs require null priorities and a pre-routing review flag. Unassigned outcomes also require human routing review.

## 10. Assigned/Unassigned decisions
I retain every valid row position, key, voicemail ID and official decision. Malformed and duplicate rows are recorded explicitly, not silently dropped.

In [2]:
LABEL_MAP = {"complaint_category": "complaint_category", "complaint_urgency": "complaint_urgency",
             "customer_state": "customer_state_text", "churn_risk": "churn_risk"}
SUPPLEMENT_FIELDS = {"schema_version", "label_version"}
REQUIRED_META = {"run_id", "model", "dataset_version", "seed", "split"}
def validate_meta(meta):
    if not isinstance(meta, dict) or REQUIRED_META - meta.keys():
        raise ValueError("Producer metadata missing: run_id, model, dataset_version, seed, split")
    if meta["split"] != "validation":
        raise ValueError("Only validation accepted; final test access forbidden")
    for name in ("run_id", "model", "dataset_version"):
        if not isinstance(meta[name], str) or not meta[name].strip():
            raise ValueError(f"Missing metadata: {name}")
    if type(meta["seed"]) is not int or meta["seed"] < 0:
        raise ValueError("seed must be nonnegative integer")
    if meta.get("schema_version") != SCHEMA_VERSION or meta.get("label_version") != LABEL_VERSION:
        raise ValueError("Explicit runtime schema_version and label_version required")
    return copy.deepcopy(meta)

def load_validation_predictions(path=None, *, approvals=None, classes_by_field=None):
    # Fail before opening ANY file, even if approval-like flags are supplied.
    raise PermissionError("Real loading disabled: verified artifacts and authorization required")

TRANSCRIPT_SOURCE = "final/clean_training_val.csv"
APPROVED_VALIDATION_SPLIT = "val"
APPROVED_VALIDATION_ROW_COUNT = 1003
APPROVED_PREDICTIONS_RUN = "fair-marbertv2-lr2e5"
APPROVED_PREDICTIONS_DIR = f"runs/exp_{APPROVED_PREDICTIONS_RUN}"

def validation_manifest_keys(manifest_rows):
    """Validate the selected val rows from the ready-branch manifest."""
    keys = set()
    for index, row in enumerate(manifest_rows):
        if not isinstance(row, dict) or row.get("split") != APPROVED_VALIDATION_SPLIT:
            raise ValueError(f"Manifest row {index} must have split val")
        audio_file = row.get("audio_file")
        if not isinstance(audio_file, str) or not audio_file.strip():
            raise ValueError(f"Manifest row {index} must have a nonempty audio_file")
        key = (row["split"], audio_file)
        if key in keys:
            raise ValueError(f"Duplicate approved manifest key: {key}")
        keys.add(key)
    return keys
DEFAULT_REVIEW_THRESHOLD = 0.5
CATEGORY_REQUESTS = {
    "network_and_connectivity": "The customer is contacting support about network connectivity.",
    "billing_and_charges": "The customer is contacting support about billing or charges.",
    "balance_and_data": "The customer is contacting support about balance or data.",
    "app_and_account": "The customer is contacting support about the app or their account.",
    "branch_and_staff": "The customer is contacting support about a branch or staff.",
    "no_reply_or_long_wait": "The customer reports a long wait or no reply.",
    "cancellation_request": "The customer is asking about cancellation.",
    "thanks_or_praise": "The customer is expressing thanks or praise.",
    "other": "The customer is contacting support about another topic.",
    "offers_and_packages": "The customer is contacting support about offers or packages.",
    "sim_services": "The customer is contacting support about SIM services.",
}

def index_transcripts(csv_rows):
    """Index parsed clean_training_val.csv rows without rewriting their text."""
    indexed = {}
    for row in csv_rows:
        key = row.get("audio_file")
        if not isinstance(key, str) or not key.strip() or key in indexed:
            raise ValueError("Transcript rows require unique, nonempty audio_file keys")
        indexed[key] = row.get("text")
    return indexed

def summarize_transcript(transcript):
    """Return the provisional first 200 Python characters of corrected text."""
    if not isinstance(transcript, str) or not transcript.strip():
        raise ValueError("Corrected text must be a nonempty string")
    return transcript[:200]

def category_customer_request(category):
    """Return a provisional category sentence, or an empty string for a null category."""
    if category is None:
        return ""
    return CATEGORY_REQUESTS[ComplaintCategory(category).value]

PROPOSED_SUGGESTED_SCRIPTS = {
    ComplaintCategory.NETWORK_AND_CONNECTIVITY:
        "ممكن توضح المشكلة بتحصل في المكالمات ولا في الإنترنت، وبتظهر في مكان معين؟",
    ComplaintCategory.BILLING_AND_CHARGES:
        "ممكن توضح أنهي بند في الفاتورة أو أنهي رسوم محتاج تستفسر عنها؟",
    ComplaintCategory.BALANCE_AND_DATA:
        "استفسارك عن الرصيد ولا عن استهلاك الإنترنت؟ ممكن توضح إيه اللي لاحظته؟",
    ComplaintCategory.APP_AND_ACCOUNT:
        "المشكلة في تسجيل الدخول ولا في استخدام خدمة جوه التطبيق؟ ممكن توضح عند أنهي خطوة ظهرت؟",
    ComplaintCategory.BRANCH_AND_STAFF:
        "شكرًا إنك شاركتنا ملاحظتك. ممكن توضح لنا تفاصيل أكتر عن ملاحظتك بخصوص الفرع أو فريق الخدمة؟",
    ComplaintCategory.NO_REPLY_OR_LONG_WAIT:
        "ممكن توضح لنا تفاصيل أكتر عن التواصل أو الانتظار، وإيه المساعدة اللي محتاجها دلوقتي؟",
    ComplaintCategory.CANCELLATION_REQUEST:
        "ممكن توضح أنهي خدمة بتسأل عن إلغائها؟ نقدر نوضح لك الخطوات والشروط الخاصة بيها.",
    ComplaintCategory.THANKS_OR_PRAISE:
        "شكرًا إنك شاركتنا تجربتك. يسعدنا نعرف إيه اللي عجبك في الخدمة.",
    ComplaintCategory.OTHER:
        "ممكن توضح موضوع استفسارك وإيه المساعدة اللي محتاجها؟",
    ComplaintCategory.OFFERS_AND_PACKAGES:
        "ممكن توضح أنهي عرض أو باقة بتسأل عنها، أو إيه أهم حاجة بتدور عليها في الباقة؟",
    ComplaintCategory.SIM_SERVICES:
        "استفسارك عن تفعيل الشريحة، استبدالها، ولا عن استخدامها؟ ممكن توضح إيه اللي محتاجه؟",
}

def suggest_script(category):
    """Look up a proposed draft for Hazem's review by enum or exact wire value."""
    return PROPOSED_SUGGESTED_SCRIPTS[ComplaintCategory(category)]

def unit_interval(value, name):
    if type(value) not in (int, float) or not 0 <= value <= 1:
        raise ValueError(f"{name} must be numeric in [0, 1]")
    return value

def adapt_row(row, supplements, meta, *, transcripts, probabilities,
              review_threshold=DEFAULT_REVIEW_THRESHOLD):
    validate_meta(meta)
    expected = {"audio_file", *LABEL_MAP}
    if not isinstance(row, dict) or set(row) != expected:
        raise ValueError(f"Unsupported row; exact keys required: {sorted(expected)}")
    key = row["audio_file"]
    if not isinstance(key, str) or not key.strip():
        raise ValueError("audio_file must be nonempty")
    extra = {}
    if supplements is not None:
        # Runtime-only fixtures can carry explicit null-label evidence and versions.
        if not isinstance(supplements, dict) or key not in supplements:
            raise ValueError("Missing runtime fixture supplemental fields for audio_file")
        extra = supplements[key]
        if (not isinstance(extra, dict) or not SUPPLEMENT_FIELDS <= extra.keys()
                or extra.keys() - (SUPPLEMENT_FIELDS | {"confidence", "suggested_script", "voicemail_id"})):
            missing = SUPPLEMENT_FIELDS - set(extra) if isinstance(extra, dict) else SUPPLEMENT_FIELDS
            raise ValueError(f"Runtime fixture supplemental fields missing/unsupported: {sorted(missing)}")
        if extra["schema_version"] != meta["schema_version"] or extra["label_version"] != meta["label_version"]:
            raise ValueError("Row and metadata versions differ")
    if key not in transcripts:
        raise ValueError("Missing corrected text for audio_file")
    transcript = transcripts[key]
    summary = summarize_transcript(transcript)
    customer_request = category_customer_request(row["complaint_category"])
    unit_interval(review_threshold, "review_threshold")
    missing_label = any(row[field] is None for field in LABEL_MAP)
    if missing_label:
        confidence = unit_interval(extra.get("confidence"), "explicit null-label confidence")
    else:
        try:
            scores = [probabilities[key][field][row[field]] for field in LABEL_MAP]
        except (KeyError, TypeError) as exc:
            raise ValueError("Missing predicted-class probability for audio_file") from exc
        confidence = min(unit_interval(score, "predicted-class probability") for score in scores)
    category = row["complaint_category"]
    if category is None:
        script = extra.get("suggested_script")
        if not isinstance(script, str) or not script.strip():
            raise ValueError("Null category requires an explicit upstream script")
    else:
        script = suggest_script(category)
    payload = dict(extra, voicemail_id=key, summary=summary, customer_request=customer_request, confidence=confidence,
                   suggested_script=script, schema_version=SCHEMA_VERSION, label_version=LABEL_VERSION,
                   needs_human_review=missing_label or confidence < review_threshold)
    payload.update({target: row[source] for source, target in LABEL_MAP.items()})
    return TextAnalysis.model_validate(payload)

def priority_expected(text):
    # Independent verification oracle, never used to produce decisions.
    if any(getattr(text, f) is None for f in ("complaint_urgency", "churn_risk", "customer_state_text")):
        return None, None
    score = {"high": 3, "medium": 2, "low": 1}[text.complaint_urgency.value]
    score += 2 if text.churn_risk.value == "at_risk" else 0
    score += int(text.customer_state_text.value in ("mad", "frustrated"))
    return score, {1: "low", 2: "low", 3: "normal", 4: "normal", 5: "high", 6: "urgent"}[score]

def reason_type(text, before):
    if text.complaint_category is None:
        return "missing_category"
    matched = [a for a in before if text.complaint_category.value in a["skills"]]
    if not matched:
        return "no_matching_skill"
    shifted = [a for a in matched if a["on_shift"]]
    return "off_shift" if not shifted else "capacity_exhausted"

def run_batch(rows, supplements, meta, *, capacity_mode, transcripts=None, probabilities=None,
              synthetic=False, manifest_rows=None, review_threshold=DEFAULT_REVIEW_THRESHOLD,
              wire_rejections=None):
    if synthetic is not True:
        raise PermissionError("Official validation execution disabled in preparation")
    if capacity_mode != "synthetic_sequential_release":
        raise ValueError("Select explicit synthetic_sequential_release test lifecycle")
    return _run_batch(rows, supplements, meta, transcripts=transcripts, probabilities=probabilities,
                      manifest_rows=manifest_rows, review_threshold=review_threshold,
                      wire_rejections=wire_rejections)

def _run_batch(rows, supplements, meta, *, transcripts, probabilities, manifest_rows,
               review_threshold, wire_rejections, real=False):
    meta = validate_meta(meta)
    unit_interval(review_threshold, "review_threshold")
    if not isinstance(transcripts, dict) or not isinstance(probabilities, dict):
        raise ValueError("Batch requires transcript and probability maps keyed by audio_file")
    if not isinstance(rows, list):
        raise ValueError("rows must be a list")
    if not isinstance(manifest_rows, list):
        raise ValueError("Batch requires approved manifest rows")
    approved_keys = validation_manifest_keys(manifest_rows)
    approved_row_count = len(approved_keys)
    # The official reader flattens rows after validating one file-level split.
    input_keys = [prediction_key(row) for row in rows]
    wire_rejections = wire_rejections or {}
    counts = Counter(key for key in input_keys if key is not None)
    missing_keys = sorted(approved_keys - counts.keys())
    unexpected_keys = sorted(counts.keys() - approved_keys)
    duplicate_keys = sorted(key for key, count in counts.items() if count > 1)
    joined_ids = {}
    if isinstance(supplements, dict):
        for key in approved_keys & counts.keys():
            extra = supplements.get(key[1])
            voicemail_id = extra.get("voicemail_id") if isinstance(extra, dict) else None
            if isinstance(voicemail_id, str) and voicemail_id.strip():
                joined_ids[key] = voicemail_id
    id_counts = Counter(joined_ids.values())
    initial, records, errors, rejections, prepared = snapshot(), [], [], [], []
    for index, (row, composite_key) in enumerate(zip(rows, input_keys, strict=True)):
        stage = "manifest_reconciliation"
        if composite_key is None:
            code, diagnostic = "invalid_prediction_key", "Missing or invalid prediction audio_file"
        elif counts[composite_key] > 1:
            code, diagnostic = "duplicate_prediction_key", "Duplicate prediction composite key"
        elif composite_key not in approved_keys:
            code, diagnostic = "unexpected_prediction_key", "Unexpected prediction composite key"
        elif index in wire_rejections:
            stage = "official_reader"
            code, diagnostic = "invalid_wire_row", wire_rejections[index]
        elif id_counts[joined_ids.get(composite_key)] > 1:
            code, diagnostic = "duplicate_legacy_id", "Duplicate voicemail_id"
        else:
            try:
                text = adapt_row(row, supplements, meta, transcripts=transcripts,
                                 probabilities=probabilities, review_threshold=review_threshold)
                prepared.append((index, composite_key[1], text))
                continue
            except (ValueError, TypeError) as exc:
                stage = "runtime_adapter"
                code, diagnostic = "invalid_runtime_row", str(exc)
        errors.append({"row_index": index, "key": composite_key, "error": diagnostic})
        rejections.append({"source_index": index, "key": composite_key, "stage": stage,
                           "reason_code": code, "diagnostic": diagnostic})
    for index, key, text in prepared:
        before = snapshot()
        eligible = [a for a in before if text.complaint_category is not None
                    and text.complaint_category.value in a["skills"] and a["on_shift"]
                    and a["current_load"] < a["capacity"]]
        decision = None
        try:
            decision = triage_phase1(text)
            expected_id = eligible[0]["id"] if eligible else None
            assert decision.assigned_agent_id == expected_id
            assert decision.assignment_status.value == ("assigned" if expected_id else "unassigned")
            assert decision.route_reason.strip()
            score, bucket = priority_expected(text)
            assert decision.priority_score == score
            assert (decision.queue_priority.value if decision.queue_priority else None) == bucket
            assert score is not None or text.needs_human_review
            expected_loads = copy.deepcopy(before)
            for a in expected_loads:
                a["current_load"] += int(a["id"] == expected_id)
            assert snapshot() == expected_loads, "Unexpected Agent mutation"
            records.append({"row_index": index, "audio_file": key, "voicemail_id": text.voicemail_id,
                "transcript": transcripts[key], "decision": decision.model_dump(mode="json"), "human_review": text.needs_human_review or expected_id is None,
                "unassigned_type": reason_type(text, before) if expected_id is None else None,
                "eligibility_verified": True, "priority_verified": True})
        finally:
            # Runner owns returned reservations; official triage owns its own failed construction.
            # No TicketStore.complete call, no second release, no reset_loads.
            if decision is not None and decision.assigned_agent_id is not None:
                agents.release(decision.assigned_agent_id)
            assert snapshot() == before, "Reservation cleanup failed"
    assert snapshot() == initial
    processed = len(records)
    assigned = sum(r["decision"]["assignment_status"] == "assigned" for r in records)
    invalid_approved = len({item["key"] for item in rejections if item["key"] in approved_keys})
    routing_unassigned = processed - assigned
    if assigned + routing_unassigned + invalid_approved + len(missing_keys) != approved_row_count:
        raise RuntimeError("Approved manifest reconciliation invariant failed")
    return {"scope": "REAL VALIDATION" if real else "SYNTHETIC ONLY", "metrics_scope": "sequential_validation",
        "metadata": meta, "capacity_mode": "sequential_release" if real else "synthetic_sequential_release",
        "review_threshold": review_threshold,
        "skill_additions_approval": "APPROVED_HAZEM", "total_input_rows": len(rows),
        "approved_manifest_rows": approved_row_count,
        "manifest_reconciliation": {"missing_keys": missing_keys, "duplicate_keys": duplicate_keys,
                                    "unexpected_keys": unexpected_keys},
        "missing_predictions": [{"key": key, "error": "Missing prediction for approved manifest key"}
                                for key in missing_keys],
        "valid_rows": len(prepared), "invalid_rows": len(errors), "processed_rows": processed,
        "input_coverage_rate": processed / len(rows) if rows else None,
        "approved_validation_coverage_rate": processed / approved_row_count if approved_row_count else None,
        "assigned_count": assigned, "unassigned_count": approved_row_count - assigned,
        "routing_unassigned_count": routing_unassigned,
        "invalid_approved_rows": invalid_approved,
        "missing_approved_rows": len(missing_keys),
        "assignment_rate": assigned / approved_row_count if approved_row_count else None,
        "assignment_rate_denominator": approved_row_count,
        "assigned_over_processed_diagnostic": assigned / processed if processed else None,
        "priority_bucket_distribution": dict(Counter(r["decision"]["queue_priority"] or "null" for r in records)),
        "null_priority_count": sum(r["decision"]["priority_score"] is None for r in records),
        "human_review_count": sum(r["human_review"] for r in records),
        "unassigned_reasons": dict(Counter(r["unassigned_type"] for r in records if r["unassigned_type"])),
        "contract_validation_errors": errors, "rejected_submitted_occurrences": rejections, "records": records,
        "official_validation_result": real or None, "target_assignment_rate": 0.9}

APPROVED_VALIDATION_SEEDS = (7, 42, 99, 1337, 2023)
APPROVED_MANIFEST_REVISION = "32e32436854cd214346ff87e7dc856897645eae0"
EXPORT_CLASSES = {
    "complaint_category": [category.value for category in ComplaintCategory],
    "complaint_urgency": [urgency.value for urgency in ComplaintUrgency],
    "customer_state": ["calm", "frustrated", "mad", "satisfied"],
    "churn_risk": [risk.value for risk in ChurnRisk],
}

class ValidationFileError(ValueError):
    """Fatal preflight failure. No seed metrics are produced."""

def unique_json_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValidationFileError(f"Duplicate JSON object field: {key}")
        result[key] = value
    return result

def prediction_key(row):
    audio_file = row.get("audio_file") if isinstance(row, dict) else None
    if isinstance(audio_file, str) and audio_file.strip():
        return APPROVED_VALIDATION_SPLIT, audio_file
    return None

def run_validation_seed(payload, *, source_path, seed, manifest_rows, manifest_revision,
                        synthetic=False, review_threshold=DEFAULT_REVIEW_THRESHOLD):
    """Exercise one seed using supplied synthetic bytes. This never opens source_path."""
    if synthetic is not True:
        raise PermissionError("Use the separately authorized Real Validation entry point")
    return _run_validation_seed(payload, source_path=source_path, seed=seed, manifest_rows=manifest_rows,
                                manifest_revision=manifest_revision, review_threshold=review_threshold)

def _run_validation_seed(payload, *, source_path, seed, manifest_rows, manifest_revision,
                         review_threshold, real=False):
    if type(seed) is not int or seed not in APPROVED_VALIDATION_SEEDS:
        raise ValidationFileError("Seed is not approved for Validation")
    expected_path = Path(APPROVED_PREDICTIONS_DIR) / f"predictions_val_seed{seed}.json"
    source_path = Path(source_path)
    if source_path.parts[-len(expected_path.parts):] != expected_path.parts:
        raise ValidationFileError(f"Expected Validation source path ending in {expected_path.as_posix()}")
    if manifest_revision != APPROVED_MANIFEST_REVISION:
        raise ValidationFileError("Unexpected approved manifest revision")
    try:
        if not isinstance(manifest_rows, list):
            raise ValueError("Expected a list of Validation manifest rows")
        approved_keys = validation_manifest_keys(manifest_rows)
    except ValueError as exc:
        raise ValidationFileError(str(exc)) from exc
    if len(approved_keys) != APPROVED_VALIDATION_ROW_COUNT:
        raise ValidationFileError("Approved manifest must contain exactly 1,003 unique val keys")
    if not isinstance(payload, bytes):
        raise ValidationFileError("Expected in-memory prediction bytes")
    try:
        envelope = json.loads(payload.decode("utf-8"), object_pairs_hook=unique_json_object)
    except (UnicodeDecodeError, json.JSONDecodeError) as exc:
        raise ValidationFileError(f"Invalid prediction JSON: {exc}") from exc
    header = {"schema_version": PREDICTIONS_SCHEMA_VERSION, "split": APPROVED_VALIDATION_SPLIT,
              "seed": seed, "run_id": APPROVED_PREDICTIONS_RUN, "model": "MARBERTv2",
              "dataset_version": "v2"}
    if not isinstance(envelope, dict) or envelope.keys() != header.keys() | {"rows"}:
        raise ValidationFileError("Unexpected prediction envelope fields")
    for name, expected in header.items():
        actual = envelope[name]
        if type(actual) is not type(expected) or actual != expected:
            raise ValidationFileError(f"Unexpected prediction header {name}: expected {expected!r}")
    original_rows = envelope["rows"]
    if not isinstance(original_rows, list) or not original_rows:
        raise ValidationFileError("Prediction envelope requires a non-empty rows list")
    keys = [prediction_key(row) for row in original_rows]
    occurrences = Counter(key for key in keys if key is not None)
    rows, probabilities, wire_rejections = [], {}, {}
    with TemporaryDirectory(prefix="svr-row-validation-") as temporary:
        candidate_path = Path(temporary) / "validation_row.json"
        for index, (row, key) in enumerate(zip(original_rows, keys, strict=True)):
            # Keep one slot per submitted occurrence so reconciliation retains source indexes.
            rows.append({"audio_file": key[1]} if key else {})
            if key is None or occurrences[key] > 1 or key not in approved_keys:
                continue
            candidate_path.write_text(json.dumps(dict(header, rows=[row])), encoding="utf-8")
            try:
                validated = read_predictions_file(candidate_path, EXPORT_CLASSES,
                                                  expected_split=APPROVED_VALIDATION_SPLIT)
            except PredictionsFileError as exc:
                wire_rejections[index] = str(exc)
                continue
            rows[index] = validated["harness_rows"][0]
            probabilities.update(validated["probabilities"])
    meta = dict(header, split="validation", schema_version=SCHEMA_VERSION, label_version=LABEL_VERSION)
    report = _run_batch(rows, None, meta, transcripts=index_transcripts(manifest_rows), probabilities=probabilities,
                        manifest_rows=manifest_rows, wire_rejections=wire_rejections,
                        review_threshold=review_threshold, real=real)
    report["provenance"] = {
        "file_path": source_path.as_posix(), "sha256": hashlib.sha256(payload).hexdigest(),
        "seed": seed, "run_id": header["run_id"], "model": header["model"],
        "dataset_version": header["dataset_version"], "manifest_revision": manifest_revision,
    }
    return report

# Real execution is opt-in and is never invoked by Run All.
APPROVED_DATA_ROOT = Path("C:/Users/RC/svr-data")
APPROVED_MANIFEST_BLOB = "b1adfcc118a766a44bdc1fbe60f372829b2dc720"
REAL_VALIDATION_AUTHORIZATION = "AUTHORIZE_PHASE2_REAL_VALIDATION"
APPROVED_ARTIFACT_SHA256 = {
    TRANSCRIPT_SOURCE: "50df253ad6506b70a262b512eff1f79562f2cd65d58143127899025ad707cb18",
    f"{APPROVED_PREDICTIONS_DIR}/predictions_val_seed7.json":
        "8c560479b2b71441eca9ed8011dc1adfe68e1c3e8029293917841d75d0853036",
    f"{APPROVED_PREDICTIONS_DIR}/predictions_val_seed42.json":
        "8caa187812034242a56ef9f3fa33a6417bcbc797a9b9b94065b2f3c3f9a038c7",
    f"{APPROVED_PREDICTIONS_DIR}/predictions_val_seed99.json":
        "8c4fcfbca9eab8ed0dc6418ca035a98ba996378957c0a15add30cef170877225",
    f"{APPROVED_PREDICTIONS_DIR}/predictions_val_seed1337.json":
        "05525ece42b95db58ca1f4d1f71fe580e6d349ecdbfc1b3e8c973ff69771befc",
    f"{APPROVED_PREDICTIONS_DIR}/predictions_val_seed2023.json":
        "e6948b2150dd7af0682d21ac897ca0cb5eb968960b08abcca5ed9de398bc3f51",
}

def approved_artifact_path(relative):
    if relative not in APPROVED_ARTIFACT_SHA256:
        raise ValidationFileError("Path is outside the approved Validation allowlist")
    path = APPROVED_DATA_ROOT / relative
    try:
        for component in (*reversed(path.parents), path):
            info = component.lstat()
            if stat.S_ISLNK(info.st_mode) or getattr(info, "st_file_attributes", 0) & stat.FILE_ATTRIBUTE_REPARSE_POINT:
                raise ValidationFileError(f"Validation artifact path contains a redirect: {component}")
        if path.resolve(strict=True) != path or not stat.S_ISREG(info.st_mode):
            raise ValidationFileError(f"Expected a regular file at the exact approved path: {path}")
    except (OSError, RuntimeError) as exc:
        raise ValidationFileError(f"Cannot verify Validation artifact path: {path}") from exc
    return path

def read_windows_artifact(path):
    """Hold each directory open without delete sharing and open the leaf without following redirects."""
    import ctypes
    import msvcrt
    from ctypes import wintypes

    kernel = ctypes.WinDLL("kernel32", use_last_error=True)
    open_handle = kernel.CreateFileW
    open_handle.argtypes = (wintypes.LPCWSTR, wintypes.DWORD, wintypes.DWORD, wintypes.LPVOID,
                            wintypes.DWORD, wintypes.DWORD, wintypes.HANDLE)
    open_handle.restype = wintypes.HANDLE
    close_handle = kernel.CloseHandle
    close_handle.argtypes = (wintypes.HANDLE,)
    get_attributes = kernel.GetFileInformationByHandleEx
    get_attributes.argtypes = (wintypes.HANDLE, ctypes.c_int, wintypes.LPVOID, wintypes.DWORD)
    open_existing = 3
    open_reparse_point = 0x00200000
    backup_semantics = 0x02000000
    attribute_tag_info = 9
    read_attributes = 0x80
    generic_read = 0x80000000
    share_read, share_write = 1, 2
    invalid_handle = ctypes.c_void_p(-1).value

    def checked_handle(component, *, directory):
        handle = open_handle(str(component), read_attributes if directory else generic_read,
                             share_read | share_write if directory else share_read,
                             None, open_existing, open_reparse_point | backup_semantics, None)
        if handle == invalid_handle:
            raise ctypes.WinError(ctypes.get_last_error())
        try:
            # FILE_ATTRIBUTE_TAG_INFO consists of two DWORDs.
            attributes = (wintypes.DWORD * 2)()
            if not get_attributes(handle, attribute_tag_info, ctypes.byref(attributes), ctypes.sizeof(attributes)):
                raise ctypes.WinError(ctypes.get_last_error())
            if (attributes[0] & stat.FILE_ATTRIBUTE_REPARSE_POINT
                    or bool(attributes[0] & stat.FILE_ATTRIBUTE_DIRECTORY) != directory):
                raise ValidationFileError(f"Unexpected file type or redirect at {component}")
        except BaseException:
            close_handle(handle)
            raise
        return handle

    with ExitStack() as parents:
        for parent in reversed(path.parents):
            parents.callback(close_handle, checked_handle(parent, directory=True))
        handle = checked_handle(path, directory=False)
        try:
            fd = msvcrt.open_osfhandle(handle, os.O_RDONLY | os.O_BINARY)
        except BaseException:
            close_handle(handle)
            raise
        with os.fdopen(fd, "rb") as stream:
            return stream.read()

def read_posix_artifact(path):
    """Walk from the root using directory descriptors without following symlinks."""
    with ExitStack() as descriptors:
        flags = os.O_RDONLY | os.O_NOFOLLOW
        directory = os.open(path.anchor, flags | os.O_DIRECTORY)
        descriptors.callback(os.close, directory)
        for part in path.parts[1:-1]:
            directory = os.open(part, flags | os.O_DIRECTORY, dir_fd=directory)
            descriptors.callback(os.close, directory)
        fd = os.open(path.name, flags | os.O_NONBLOCK, dir_fd=directory)
        with os.fdopen(fd, "rb") as stream:
            if not stat.S_ISREG(os.fstat(stream.fileno()).st_mode):
                raise ValidationFileError("Expected a regular Validation artifact")
            return stream.read()

def authenticated_artifact(relative):
    path = approved_artifact_path(relative)
    try:
        payload = read_windows_artifact(path) if os.name == "nt" else read_posix_artifact(path)
    except OSError as exc:
        raise ValidationFileError(f"Cannot safely open Validation artifact: {path}") from exc
    if hashlib.sha256(payload).hexdigest() != APPROVED_ARTIFACT_SHA256[relative]:
        raise ValidationFileError(f"SHA-256 mismatch for {relative}")
    return path, payload

def authenticated_manifest(payload):
    if (git(APPROVED_DATA_ROOT, "rev-parse", "HEAD") != APPROVED_MANIFEST_REVISION
            or git(APPROVED_DATA_ROOT, "rev-parse", f"{APPROVED_MANIFEST_REVISION}:{TRANSCRIPT_SOURCE}")
            != APPROVED_MANIFEST_BLOB):
        raise ValidationFileError("Approved manifest Git revision or tracked blob mismatch")
    # The SHA-256 authenticates exact CRLF working bytes. Git stores the LF-normalized blob.
    normalized = payload.replace(b"\r\n", b"\n")
    git_object_header = f"blob {len(normalized)}\0".encode("ascii")
    if hashlib.sha1(git_object_header + normalized).hexdigest() != APPROVED_MANIFEST_BLOB:
        raise ValidationFileError("Manifest bytes do not match the approved Git blob")
    try:
        with io.StringIO(payload.decode("utf-8-sig"), newline="") as stream:
            reader = csv.DictReader(stream, strict=True)
            headers = reader.fieldnames or []
            if len(headers) != len(set(headers)) or not {"split", "audio_file", "text"} <= set(headers):
                raise ValidationFileError("Manifest header has missing or duplicate fields")
            rows = list(reader)
    except (UnicodeDecodeError, csv.Error) as exc:
        raise ValidationFileError("Invalid approved manifest CSV") from exc
    if (len(validation_manifest_keys(rows)) != APPROVED_VALIDATION_ROW_COUNT
            or any(None in row or None in row.values() or not row["text"].strip() for row in rows)):
        raise ValidationFileError("Manifest requires 1,003 complete and unique val keys with corrected text")
    return rows

def run_authorized_validation(*, authorization=None, review_threshold=DEFAULT_REVIEW_THRESHOLD):
    """Run the five approved seeds only after explicit authorization and artifact authentication."""
    if type(authorization) is not str or authorization != REAL_VALIDATION_AUTHORIZATION:
        raise PermissionError("Separate explicit authorization is required for Real Validation")
    if not __debug__ or snapshot() != BASELINE:
        raise RuntimeError("Real Validation requires assertions and baseline Agent State")
    unit_interval(review_threshold, "review_threshold")
    manifest_path, manifest_bytes = authenticated_artifact(TRANSCRIPT_SOURCE)
    manifest_rows = authenticated_manifest(manifest_bytes)
    sources = {
        seed: authenticated_artifact(f"{APPROVED_PREDICTIONS_DIR}/predictions_val_seed{seed}.json")
        for seed in APPROVED_VALIDATION_SEEDS
    }
    reports = {}
    try:
        for seed, (path, payload) in sources.items():
            report = _run_validation_seed(
                payload, source_path=path, seed=seed, manifest_rows=manifest_rows,
                manifest_revision=APPROVED_MANIFEST_REVISION, review_threshold=review_threshold, real=True,
            )
            report["provenance"].update(manifest_file_path=manifest_path.as_posix(),
                                        manifest_sha256=hashlib.sha256(manifest_bytes).hexdigest(),
                                        manifest_git_blob=APPROVED_MANIFEST_BLOB)
            reports[seed] = report
    finally:
        if snapshot() != BASELINE:
            raise RuntimeError("Real Validation failed to restore Agent State")
    return reports

META = {"run_id": "synthetic-preparation", "model": "synthetic-fixture-v1",
        "dataset_version": "synthetic-only", "seed": 42, "split": "validation",
        "schema_version": 1, "label_version": LABEL_VERSION}
def fixture(key="synthetic-001", **labels):
    row = {"audio_file": key, "complaint_category": "network_and_connectivity",
           "complaint_urgency": "medium", "customer_state": "frustrated", "churn_risk": "not_at_risk"}
    row.update(labels)
    supplement = {"voicemail_id": "vm-" + key, "suggested_script": "Synthetic draft script",
        "confidence": 0.9,
        "schema_version": 1, "label_version": LABEL_VERSION}
    return row, {key: supplement}
def run_one(**labels):
    row, supplemental = fixture(**labels)
    return synthetic_batch([row], supplemental, META, capacity_mode="synthetic_sequential_release", synthetic=True)

SYNTHETIC_TRANSCRIPT = "  Corrected transcript with punctuation! "
def fixture_inputs(rows):
    return {
        "transcripts": index_transcripts({"audio_file": row["audio_file"], "text": SYNTHETIC_TRANSCRIPT}
                                         for row in rows if isinstance(row, dict)
                                         and isinstance(row.get("audio_file"), str)),
        "probabilities": {row["audio_file"]: {field: {row.get(field): 0.9} for field in LABEL_MAP}
                          for row in rows if isinstance(row, dict) and isinstance(row.get("audio_file"), str)},
    }

def adapt_fixture(row, supplements, meta):
    return adapt_row(row, supplements, meta, **fixture_inputs([row]))

def synthetic_batch(rows, supplements, meta, **kwargs):
    unique = {row["audio_file"]: row for row in rows
              if isinstance(row, dict) and isinstance(row.get("audio_file"), str)}
    kwargs.setdefault("manifest_rows", fixture_manifest(supplements))
    return run_batch(rows, supplements, meta, **fixture_inputs(unique.values()), **kwargs)

def fixture_manifest(audio_files):
    return [{"split": APPROVED_VALIDATION_SPLIT, "audio_file": key} for key in audio_files]

## 11. Assignment Rate calculation
Assignment Rate = successfully assigned approved rows / all approved manifest rows. Hazem confirmed 1,003 for the official run. I use explicit val manifest fixtures for synthetic runs. I count approved rows with missing or invalid predictions as Unassigned and distinguish them from valid routing outcomes. I exclude unexpected inputs from the numerator and denominator and record an explicit error for every rejected input. Input Coverage uses all submitted rows. Approved Validation Coverage uses all approved manifest rows.

## 12. Unassigned Reason Analysis
I group pre-reservation evidence into missing_category, no_matching_skill, off_shift and capacity_exhausted. Every Unassigned row retains the official nonempty route_reason.

## 13. Summary Report
I demonstrate one synthetic row per category. Hazem approved sim_services for Network Agent and offers_and_packages for Billing Agent. Synthetic coverage or assignment does not establish the official 90% target.

In [3]:
rows, supplements = [], {}
for i, category in enumerate(ComplaintCategory):
    row, extra = fixture(f"synthetic-{i:03d}", complaint_category=category.value)
    rows.append(row); supplements.update(extra)
SUMMARY = synthetic_batch(rows, supplements, META, capacity_mode="synthetic_sequential_release", synthetic=True)
print(json.dumps({k: v for k, v in SUMMARY.items() if k != "records"}, indent=2))
assert snapshot() == BASELINE
print("Synthetic sequential-validation metrics.")

{
  "scope": "SYNTHETIC ONLY",
  "metrics_scope": "sequential_validation",
  "metadata": {
    "run_id": "synthetic-preparation",
    "model": "synthetic-fixture-v1",
    "dataset_version": "synthetic-only",
    "seed": 42,
    "split": "validation",
    "schema_version": 1,
    "label_version": "taxonomy-2026-11-15"
  },
  "capacity_mode": "synthetic_sequential_release",
  "review_threshold": 0.5,
  "skill_additions_approval": "APPROVED_HAZEM",
  "total_input_rows": 11,
  "approved_manifest_rows": 11,
  "manifest_reconciliation": {
    "missing_keys": [],
    "duplicate_keys": [],
    "unexpected_keys": []
  },
  "missing_predictions": [],
  "valid_rows": 11,
  "invalid_rows": 0,
  "processed_rows": 11,
  "input_coverage_rate": 1.0,
  "approved_validation_coverage_rate": 1.0,
  "assigned_count": 11,
  "unassigned_count": 0,
  "routing_unassigned_count": 0,
  "invalid_approved_rows": 0,
  "missing_approved_rows": 0,
  "assignment_rate": 1.0,
  "assignment_rate_denominator": 11,
  "assi

## 14. Automated Tests
I use stdlib unittest and controlled in-memory fixtures restored after each context. No official seed or source file is edited. Subcases are additional checks, not separate test methods.

In [4]:
class PredictionHandoffTests(unittest.TestCase):
    def setUp(self):
        self.before = snapshot()
        temporary = self.enterContext(TemporaryDirectory(prefix="svr-synthetic-predictions-"))
        self.path = Path(temporary) / "synthetic.json"
        self.classes = {
            "complaint_category": [item.value for item in ComplaintCategory],
            "complaint_urgency": [item.value for item in ComplaintUrgency],
            "customer_state": [item.value for item in CustomerState],
            "churn_risk": [item.value for item in ChurnRisk],
        }
        self.header = {"schema_version": PREDICTIONS_SCHEMA_VERSION, "split": "val",
                       "run_id": "synthetic-reader", "model": "synthetic-model-v2",
                       "seed": 1337, "dataset_version": "synthetic-dataset-v2"}
        self.data = dict(self.header, rows=[])
        self.supplements = {}
        for key, category, winning in (
            ("synthetic-z.wav", "network_and_connectivity", 0.65),
            ("synthetic-a.wav", "billing_and_charges", 0.85),
        ):
            row, extra = fixture(key, complaint_category=category)
            labels = {field: row[field] for field in LABEL_MAP}
            probabilities = {
                field: {label: winning if label == labels[field] else (1 - winning) / (len(classes) - 1)
                        for label in classes}
                for field, classes in self.classes.items()
            }
            self.data["rows"].append({"audio_file": key, "labels": labels, "probabilities": probabilities})
            self.supplements.update(extra)
        self.transcripts = index_transcripts([
            {"audio_file": "synthetic-a.wav", "text": "  Second corrected text! "},
            {"audio_file": "synthetic-z.wav", "text": "First corrected text."},
        ])

    def tearDown(self):
        self.assertEqual(snapshot(), self.before)

    def read_fixture(self, data):
        self.path.write_text(json.dumps(data), encoding="utf-8")
        return read_predictions_file(self.path, self.classes, expected_split="val")

    def runtime_meta(self, result):
        # The validated wire version and the runtime version are separate contracts.
        return dict(result["meta"], split="validation", schema_version=SCHEMA_VERSION,
                    label_version=LABEL_VERSION)

    def run_handoff(self, result, **overrides):
        options = dict(transcripts=self.transcripts, probabilities=result["probabilities"],
                       manifest_rows=fixture_manifest(self.supplements),
                       capacity_mode="synthetic_sequential_release", synthetic=True)
        options.update(overrides)
        return run_batch(result["harness_rows"], self.supplements, self.runtime_meta(result), **options)

    def test_official_reader_to_adapter_handoff(self):
        result = self.read_fixture(self.data)
        self.assertEqual(result["meta"], dict(self.header, n_rows=2))
        self.assertEqual(result["harness_rows"], [
            {"audio_file": row["audio_file"], **row["labels"]} for row in self.data["rows"]
        ])
        self.assertEqual(result["probabilities"], {
            row["audio_file"]: row["probabilities"] for row in self.data["rows"]
        })
        for probabilities in result["probabilities"].values():
            self.assertEqual(set(probabilities), set(self.classes))
            for field, classes in self.classes.items():
                self.assertEqual(set(probabilities[field]), set(classes))
        meta = self.runtime_meta(result)
        self.assertEqual(meta["schema_version"], SCHEMA_VERSION)
        self.assertEqual(meta["label_version"], LABEL_VERSION)
        self.assertEqual(meta["split"], "validation")
        for row, expected in zip(result["harness_rows"], (0.65, 0.85), strict=True):
            text = adapt_row(row, self.supplements, meta, transcripts=self.transcripts,
                             probabilities=result["probabilities"])
            self.assertEqual(text.confidence, expected)
        report = self.run_handoff(result)
        self.assertEqual(report["metadata"], meta)
        self.assertEqual(report["contract_validation_errors"], [])
        self.assertEqual([record["audio_file"] for record in report["records"]],
                         ["synthetic-z.wav", "synthetic-a.wav"])
        self.assertEqual([record["transcript"] for record in report["records"]],
                         ["First corrected text.", "  Second corrected text! "])

    def test_official_reader_rejects_synthetic_test_split_for_validation(self):
        with self.assertRaisesRegex(PredictionsFileError, "split 'test' but 'val' was expected"):
            self.read_fixture(dict(self.data, split="test"))

    def test_wire_and_runtime_version_checks_are_independent(self):
        with self.assertRaisesRegex(PredictionsFileError, "unsupported schema_version"):
            self.read_fixture(dict(self.data, schema_version=PREDICTIONS_SCHEMA_VERSION + 1))
        result = self.read_fixture(self.data)
        for field, bad in (("schema_version", SCHEMA_VERSION + 1), ("label_version", "synthetic-old")):
            for missing in (False, True):
                with self.subTest(field=field, missing=missing):
                    meta = self.runtime_meta(result)
                    if missing:
                        del meta[field]
                    else:
                        meta[field] = bad
                    with self.assertRaisesRegex(ValueError, "Explicit runtime schema_version and label_version"):
                        validate_meta(meta)
        bad = copy.deepcopy(self.supplements)
        bad["synthetic-z.wav"]["schema_version"] = SCHEMA_VERSION + 1
        with self.assertRaisesRegex(ValueError, "Row and metadata versions differ"):
            adapt_row(result["harness_rows"][0], bad, self.runtime_meta(result),
                      transcripts=self.transcripts, probabilities=result["probabilities"])

    def test_official_reader_rejects_missing_and_extra_fields(self):
        for path, field, error in (
            ((), "seed", "must have exactly the keys"),
            (("rows", 0), "labels", "must have exactly the keys"),
            (("rows", 0, "labels"), "churn_risk", "labels must have exactly the fields"),
            (("rows", 0, "probabilities"), "churn_risk", "probabilities are forced for every field"),
            (("rows", 0, "probabilities", "churn_risk"), "at_risk", "must name exactly the classes"),
        ):
            for missing in (True, False):
                with self.subTest(path=path, missing=missing):
                    data = copy.deepcopy(self.data)
                    target = data
                    for key in path:
                        target = target[key]
                    if missing:
                        del target[field]
                    else:
                        target["synthetic_extra"] = "unexpected"
                    with self.assertRaisesRegex(PredictionsFileError, error):
                        self.read_fixture(data)

    def test_official_reader_checks_seed_and_class_lists(self):
        for seed in (-1, True, "1337"):
            with self.subTest(seed=seed), self.assertRaisesRegex(PredictionsFileError, "seed must"):
                self.read_fixture(dict(self.data, seed=seed))
        data = copy.deepcopy(self.data)
        data["rows"][0]["labels"]["complaint_category"] = "synthetic-unsupported"
        with self.assertRaisesRegex(PredictionsFileError, "not in the class list"):
            self.read_fixture(data)

    def test_duplicate_ids_at_wire_and_runtime_boundaries(self):
        duplicate = dict(self.data, rows=[self.data["rows"][0]] * 2)
        with self.assertRaisesRegex(PredictionsFileError, "appears more than once"):
            self.read_fixture(duplicate)
        result = self.read_fixture(self.data)
        self.supplements["synthetic-a.wav"]["voicemail_id"] = self.supplements["synthetic-z.wav"]["voicemail_id"]
        report = self.run_handoff(result)
        self.assertEqual(report["contract_validation_errors"], [
            {"row_index": index, "key": ("val", key), "error": "Duplicate voicemail_id"}
            for index, key in enumerate(self.supplements)
        ])

    def test_unexpected_prediction_id_and_missing_transcript_are_indexed(self):
        data = copy.deepcopy(self.data)
        data["rows"][0]["audio_file"] = "synthetic-unexpected.wav"
        result = self.read_fixture(data)
        report = self.run_handoff(result)
        self.assertEqual(report["contract_validation_errors"], [
            {"row_index": 0, "key": ("val", "synthetic-unexpected.wav"), "error": "Unexpected prediction composite key"}
        ])
        self.assertEqual([record["row_index"] for record in report["records"]], [1])
        result = self.read_fixture(self.data)
        report = self.run_handoff(result, transcripts={"synthetic-a.wav": self.transcripts["synthetic-a.wav"]})
        self.assertEqual(report["contract_validation_errors"], [
            {"row_index": 0, "key": ("val", "synthetic-z.wav"), "error": "Missing corrected text for audio_file"}
        ])
        self.assertEqual([record["row_index"] for record in report["records"]], [1])

class ApprovedValidationSeedTests(unittest.TestCase):
    def setUp(self):
        self.before = snapshot()
        self.keys = [f"synthetic-{i:04d}.wav" for i in range(APPROVED_VALIDATION_ROW_COUNT)]
        self.manifest = [dict(item, text="  Corrected synthetic text! ") for item in fixture_manifest(self.keys)]

    def tearDown(self):
        self.assertEqual(snapshot(), self.before)

    def row(self, index, **overrides):
        labels = {"complaint_category": "sim_services", "complaint_urgency": "medium",
                  "customer_state": "frustrated", "churn_risk": "at_risk"}
        labels.update(overrides)
        probabilities = {
            field: {label: 0.8 if label == labels[field] else 0.2 / (len(classes) - 1) for label in classes}
            for field, classes in EXPORT_CLASSES.items()
        }
        return {"audio_file": self.keys[index], "labels": labels, "probabilities": probabilities}

    def envelope(self, seed, rows):
        return {"schema_version": PREDICTIONS_SCHEMA_VERSION, "split": "val", "seed": seed,
                "run_id": APPROVED_PREDICTIONS_RUN, "model": "MARBERTv2", "dataset_version": "v2", "rows": rows}

    def run_seed(self, seed, data, **overrides):
        options = dict(source_path=f"{APPROVED_PREDICTIONS_DIR}/predictions_val_seed{seed}.json",
                       seed=seed, manifest_rows=self.manifest, manifest_revision=APPROVED_MANIFEST_REVISION,
                       synthetic=True)
        options.update(overrides)
        payload = data if isinstance(data, bytes) else json.dumps(data).encode("utf-8")
        return run_validation_seed(payload, **options)

    def test_four_class_validation_export_preserves_runtime_and_argmax_churn(self):
        self.assertIn("unclear", {state.value for state in CustomerState})
        self.assertEqual(EXPORT_CLASSES["customer_state"], ["calm", "frustrated", "mad", "satisfied"])
        row = self.row(0)
        row["probabilities"]["churn_risk"] = {"at_risk": 0.51, "not_at_risk": 0.49}
        with patch.dict(globals(), {"triage_phase1": unittest.mock.Mock(wraps=triage_phase1)}) as patched:
            report = self.run_seed(42, self.envelope(42, [row]))
            text = patched["triage_phase1"].call_args.args[0]
        self.assertEqual(text.voicemail_id, row["audio_file"])
        self.assertEqual((text.schema_version, text.label_version), (SCHEMA_VERSION, LABEL_VERSION))
        self.assertEqual(text.churn_risk.value, "at_risk")
        self.assertEqual(text.confidence, 0.51)
        self.assertFalse(text.needs_human_review)
        self.assertEqual(report["records"][0]["decision"]["priority_score"], 5)
        self.assertEqual(report["records"][0]["transcript"], "  Corrected synthetic text! ")

    def test_five_seeds_keep_independent_reports_and_provenance(self):
        reports = {}
        for index, seed in enumerate((7, 42, 99, 1337, 2023)):
            row = self.row(index, complaint_category="offers_and_packages")
            payload = json.dumps(self.envelope(seed, [row])).encode("utf-8")
            report = self.run_seed(seed, payload)
            reports[seed] = report
            self.assertEqual(report["provenance"], {
                "file_path": f"runs/exp_fair-marbertv2-lr2e5/predictions_val_seed{seed}.json",
                "sha256": hashlib.sha256(payload).hexdigest(), "seed": seed,
                "run_id": "fair-marbertv2-lr2e5", "model": "MARBERTv2", "dataset_version": "v2",
                "manifest_revision": APPROVED_MANIFEST_REVISION,
            })
            self.assertEqual(report["assignment_rate_denominator"], 1003)
            self.assertEqual(report["assignment_rate"], 1 / 1003)
            self.assertEqual(report["missing_approved_rows"], 1002)
            self.assertEqual(report["records"][0]["audio_file"], self.keys[index])
            self.assertEqual(report["records"][0]["decision"]["assigned_agent_id"], "agent-demo-billing")
            self.assertEqual(snapshot(), self.before)
        self.assertEqual(len({report["provenance"]["sha256"] for report in reports.values()}), 5)

    def test_fatal_file_failures_stop_before_official_reader_or_routing(self):
        data = self.envelope(42, [self.row(0)])
        with patch.dict(globals(), {
            "read_predictions_file": unittest.mock.Mock(side_effect=AssertionError("Unexpected reader call")),
            "triage_phase1": unittest.mock.Mock(side_effect=AssertionError("Unexpected routing call")),
        }):
            for field, value in (("split", "test"), ("seed", 7), ("schema_version", True),
                                 ("run_id", "wrong"), ("model", "wrong"), ("dataset_version", "wrong"),
                                 ("rows", []), ("rows", {}), ("extra", True)):
                with self.subTest(field=field, value=value), self.assertRaises(ValidationFileError):
                    self.run_seed(42, dict(data, **{field: value}))
            for bad, options in (
                (b"{", {}), (b'{"split":"val","split":"test"}', {}),
                (data, {"manifest_revision": "wrong"}), (data, {"manifest_rows": self.manifest[:-1]}),
                (data, {"source_path": f"{APPROVED_PREDICTIONS_DIR}/predictions_test_seed42.json"}),
            ):
                with self.subTest(options=options.keys()), self.assertRaises(ValidationFileError):
                    self.run_seed(42, bad, **options)
            with self.assertRaises(PermissionError):
                self.run_seed(42, data, synthetic=False)

    def test_recoverable_row_rejections_retain_indexes_and_manifest_metrics(self):
        good, duplicate, bad_probability, bad_label, later, no_text = [self.row(i) for i in range(6)]
        bad_probability["probabilities"]["churn_risk"]["at_risk"] = 1.01
        bad_label["labels"]["customer_state"] = "unclear"
        del self.manifest[5]["text"]
        unexpected = dict(good, audio_file="synthetic-unexpected.wav")
        rows = [bad_probability, duplicate, good, bad_label, duplicate, None, unexpected, no_text, later]
        with patch.object(agents, "release", wraps=agents.release) as release:
            report = self.run_seed(42, self.envelope(42, rows))
        self.assertEqual([row["row_index"] for row in report["records"]], [2, 8])
        self.assertEqual([call.args for call in release.call_args_list], [("agent-demo-network",)] * 2)
        self.assertEqual(report["assigned_count"], 2)
        self.assertEqual(report["routing_unassigned_count"], 0)
        self.assertEqual(report["invalid_approved_rows"], 4)
        self.assertEqual(report["missing_approved_rows"], 997)
        self.assertEqual(report["invalid_rows"], 7)
        self.assertEqual(report["assignment_rate"], 2 / 1003)
        self.assertEqual(report["approved_validation_coverage_rate"], 2 / 1003)
        self.assertEqual(report["input_coverage_rate"], 2 / 9)
        self.assertEqual(report["manifest_reconciliation"]["duplicate_keys"], [("val", duplicate["audio_file"])])
        ledger = report["rejected_submitted_occurrences"]
        self.assertEqual([(item["source_index"], item["reason_code"]) for item in ledger], [
            (0, "invalid_wire_row"), (1, "duplicate_prediction_key"), (3, "invalid_wire_row"),
            (4, "duplicate_prediction_key"), (5, "invalid_prediction_key"),
            (6, "unexpected_prediction_key"), (7, "invalid_runtime_row"),
        ])
        self.assertEqual(ledger[0]["stage"], "official_reader")
        self.assertEqual(ledger[0]["key"], ("val", bad_probability["audio_file"]))
        self.assertIn("between 0 and 1", ledger[0]["diagnostic"])
        self.assertIn("not in the class list", ledger[2]["diagnostic"])
        self.assertIsNone(ledger[4]["key"])
        self.assertEqual(ledger[-1]["stage"], "runtime_adapter")
        self.assertIn("Corrected text must be a nonempty string", ledger[-1]["diagnostic"])

    def test_valid_envelope_with_only_invalid_rows_scores_zero(self):
        row = self.row(0)
        row["labels"]["churn_risk"] = "not_at_risk"
        report = self.run_seed(7, self.envelope(7, [row]))
        self.assertEqual(report["assignment_rate"], 0.0)
        self.assertEqual(report["unassigned_count"], 1003)
        self.assertEqual(report["invalid_approved_rows"], 1)
        self.assertIn("most probable class", report["rejected_submitted_occurrences"][0]["diagnostic"])

    def test_missing_and_null_wire_labels_and_missing_probabilities_are_rejected(self):
        missing_label, null_label, missing_probability, accepted = [self.row(i) for i in range(4)]
        del missing_label["labels"]["churn_risk"]
        null_label["labels"]["churn_risk"] = None
        del missing_probability["probabilities"]["churn_risk"]["at_risk"]
        report = self.run_seed(42, self.envelope(42, [missing_label, null_label, missing_probability, accepted]))
        self.assertEqual([record["row_index"] for record in report["records"]], [3])
        self.assertEqual(report["invalid_approved_rows"], 3)
        self.assertEqual(report["assignment_rate"], 1 / 1003)
        for index, phrase in enumerate(("labels must have exactly", "not in the class list",
                                        "must name exactly the classes")):
            rejection = report["rejected_submitted_occurrences"][index]
            self.assertEqual(rejection["source_index"], index)
            self.assertEqual(rejection["stage"], "official_reader")
            self.assertEqual(rejection["reason_code"], "invalid_wire_row")
            self.assertIn(phrase, rejection["diagnostic"])

    def test_high_confidence_unassigned_validation_ticket_requires_final_review(self):
        from svr.schemas import AudioEvidence, TriageDecision

        row = self.row(0)
        triage = unittest.mock.Mock(wraps=triage_phase1)
        with (
            patch.dict(globals(), {"triage_phase1": triage}),
            patch.object(agents, "SEEDED_AGENTS", []),
            patch.object(agents, "release", wraps=agents.release) as release,
        ):
            report = self.run_seed(42, self.envelope(42, [row]), review_threshold=0.7)
            release.assert_not_called()
        self.assertEqual(snapshot(), self.before)
        record = report["records"][0]
        text = triage.call_args.args[0]
        self.assertEqual(text.confidence, 0.8)
        self.assertFalse(text.needs_human_review)
        self.assertTrue(record["human_review"])
        self.assertEqual(report["routing_unassigned_count"], 1)
        decision = TriageDecision.model_validate(record["decision"])
        audio = AudioEvidence(voicemail_id=text.voicemail_id, format="wav", transcript=record["transcript"],
                              stt_provider="synthetic", stt_model_version="synthetic-v1")
        ticket = assemble_ticket(ticket_id="synthetic-unassigned", voicemail_id=text.voicemail_id,
                                 audio=audio, text=text, voice_evidence=None, decision=decision,
                                 stage_status={"triage": "ok"}, stage_timing_ms={"triage": 0.0},
                                 processing_ms=0.0)
        self.assertTrue(ticket.needs_human_review)
        self.assertEqual(ticket.ticket_status.value, "needs_review")
        self.assertEqual(ticket.transcript, self.manifest[0]["text"])

    def test_seed_failure_releases_reservation_and_restores_state(self):
        row = self.row(0)
        with (
            patch.dict(globals(), {"priority_expected": lambda _: (-1, None)}),
            patch.object(agents, "release", wraps=agents.release) as release,
        ):
            with self.assertRaises(AssertionError):
                self.run_seed(42, self.envelope(42, [row]))
            release.assert_called_once_with("agent-demo-network")
            self.assertEqual(snapshot(), self.before)

class GatedValidationLoaderTests(unittest.TestCase):
    def setUp(self):
        self.before = snapshot()
        self.root = Path(self.enterContext(TemporaryDirectory(prefix="svr-loader-fixture-"))).resolve()
        self.keys = [f"synthetic-loader-{i}.wav" for i in range(1003)]
        self.manifest = ("split,audio_file,text\r\n" +
                         "".join(f'val,{key},"  Corrected text {i}!  "\r\n'
                                 for i, key in enumerate(self.keys))).encode("utf-8")
        self.paths = {seed: f"{APPROVED_PREDICTIONS_DIR}/predictions_val_seed{seed}.json"
                      for seed in (7, 42, 99, 1337, 2023)}
        self.assertEqual(set(APPROVED_ARTIFACT_SHA256), {TRANSCRIPT_SOURCE, *self.paths.values()})
        self.payloads = {TRANSCRIPT_SOURCE: self.manifest}
        for index, (seed, relative) in enumerate(self.paths.items()):
            labels = {"complaint_category": "sim_services", "complaint_urgency": "medium",
                      "customer_state": "calm", "churn_risk": "not_at_risk"}
            probabilities = {
                field: {label: 0.8 if label == labels[field] else 0.2 / (len(classes) - 1)
                        for label in classes}
                for field, classes in EXPORT_CLASSES.items()
            }
            data = {"schema_version": 1, "split": "val", "seed": seed,
                    "run_id": APPROVED_PREDICTIONS_RUN, "model": "MARBERTv2", "dataset_version": "v2",
                    "rows": [{"audio_file": self.keys[index], "labels": labels, "probabilities": probabilities}]}
            self.payloads[relative] = json.dumps(data).encode("utf-8")
        self.digests = {path: hashlib.sha256(payload).hexdigest() for path, payload in self.payloads.items()}
        for relative, payload in self.payloads.items():
            path = self.root / relative
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(payload)
        self.blob = self.fixture_blob(self.manifest)
        self.git = unittest.mock.Mock(
            side_effect=lambda _, *args: (APPROVED_MANIFEST_REVISION if args[-1] == "HEAD" else self.blob)
        )
        self.enterContext(patch.dict(globals(), {
            "APPROVED_DATA_ROOT": self.root, "APPROVED_ARTIFACT_SHA256": self.digests,
            "APPROVED_MANIFEST_BLOB": self.blob, "git": self.git,
        }))

    def tearDown(self):
        self.assertEqual(snapshot(), self.before)

    def fixture_blob(self, payload):
        payload = payload.replace(b"\r\n", b"\n")
        return hashlib.sha1(f"blob {len(payload)}\0".encode("ascii") + payload).hexdigest()

    def authorize_fixture(self):
        return run_authorized_validation(authorization=REAL_VALIDATION_AUTHORIZATION)

    def test_loader_requires_explicit_authorization_before_any_data_access(self):
        with patch.dict(globals(), {"authenticated_artifact": unittest.mock.Mock(
            side_effect=AssertionError("Artifact access without authorization")
        )}):
            for denied in (None, True, "yes"):
                with self.subTest(authorization=denied), self.assertRaises(PermissionError):
                    run_authorized_validation(authorization=denied)
        self.git.assert_not_called()

    def test_exact_allowlist_rejects_unapproved_paths_before_opening(self):
        relative = self.paths[42]
        path, payload = authenticated_artifact(relative)
        self.assertEqual(path, self.root / relative)
        self.assertEqual(payload, self.payloads[relative])
        with patch.object(Path, "lstat", side_effect=AssertionError("Inspected an unapproved path")):
            for denied in ("final/clean_training_test.csv", "runs/exp_fair-marbertv2-lr2e5/predictions_test_seed42.json",
                           "final/../final/clean_training_val.csv", str(path), relative.upper()):
                with self.subTest(path=denied), self.assertRaisesRegex(ValidationFileError, "allowlist"):
                    authenticated_artifact(denied)

    def test_redirection_is_rejected_before_artifact_read(self):
        real_lstat = Path.lstat
        redirected = self.root / "final"

        def lstat(path, *args, **kwargs):
            if path == redirected:
                return type("Redirect", (), {"st_mode": stat.S_IFDIR,
                                            "st_file_attributes": stat.FILE_ATTRIBUTE_REPARSE_POINT})()
            return real_lstat(path, *args, **kwargs)

        with (
            patch.object(Path, "lstat", lstat),
            patch.dict(globals(), {"read_windows_artifact": unittest.mock.Mock(side_effect=AssertionError),
                                   "read_posix_artifact": unittest.mock.Mock(side_effect=AssertionError)}),
        ):
            with self.assertRaisesRegex(ValidationFileError, "redirect"):
                authenticated_artifact(TRANSCRIPT_SOURCE)

    def test_safe_open_rejects_redirect_introduced_after_path_check(self):
        path = approved_artifact_path(TRANSCRIPT_SOURCE)
        original = path.parent
        displaced = self.root / "displaced-fixture-directory"
        original.rename(displaced)
        if os.name == "nt":
            subprocess.run(["cmd.exe", "/d", "/c", "mklink", "/J", str(original), str(displaced)],
                           check=True, capture_output=True)
        else:
            original.symlink_to(displaced, target_is_directory=True)
        try:
            # Simulate replacement between lexical preflight and the descriptor/handle open.
            with patch.dict(globals(), {"approved_artifact_path": lambda _: path}):
                with self.assertRaises(ValidationFileError):
                    authenticated_artifact(TRANSCRIPT_SOURCE)
        finally:
            if os.name == "nt":
                original.rmdir()
            else:
                original.unlink()

    def test_changed_manifest_or_prediction_hash_stops_before_routing(self):
        with patch.dict(globals(), {"triage_phase1": unittest.mock.Mock(side_effect=AssertionError)}):
            for relative in (TRANSCRIPT_SOURCE, self.paths[2023]):
                with self.subTest(path=relative):
                    path = self.root / relative
                    path.write_bytes(self.payloads[relative] + b" ")
                    with self.assertRaisesRegex(ValidationFileError, "SHA-256 mismatch"):
                        self.authorize_fixture()
                    path.write_bytes(self.payloads[relative])

    def test_manifest_git_revision_and_blob_are_verified(self):
        for revision, blob in (("wrong-revision", self.blob), (APPROVED_MANIFEST_REVISION, "wrong-blob")):
            with self.subTest(revision=revision, blob=blob):
                self.git.side_effect = [revision, blob]
                with self.assertRaisesRegex(ValidationFileError, "revision or tracked blob"):
                    self.authorize_fixture()
        self.git.side_effect = [APPROVED_MANIFEST_REVISION, self.blob]
        with self.assertRaisesRegex(ValidationFileError, "bytes do not match"):
            authenticated_manifest(self.manifest.replace(b"Corrected", b"Different"))

    def test_authenticated_manifest_rejects_invalid_keys_or_missing_text(self):
        changes = (
            (b"val,synthetic-loader-1.wav", b"val,synthetic-loader-0.wav", "Duplicate approved manifest key"),
            (b"val,synthetic-loader-1.wav", b"train,synthetic-loader-1.wav", "must have split val"),
            (b"  Corrected text 1!  ", b"   ", "with corrected text"),
        )
        for before, after, diagnostic in changes:
            payload = self.manifest.replace(before, after, 1)
            blob = self.fixture_blob(payload)
            self.git.side_effect = [APPROVED_MANIFEST_REVISION, blob]
            with self.subTest(diagnostic=diagnostic), patch.dict(globals(), {"APPROVED_MANIFEST_BLOB": blob}):
                with self.assertRaisesRegex(ValueError, diagnostic):
                    authenticated_manifest(payload)

    def test_correct_digest_does_not_override_wire_seed_validation(self):
        relative = self.paths[7]
        data = json.loads(self.payloads[relative])
        data["seed"] = 42
        payload = json.dumps(data).encode("utf-8")
        (self.root / relative).write_bytes(payload)
        self.digests[relative] = hashlib.sha256(payload).hexdigest()
        with (
            patch.dict(globals(), {"triage_phase1": unittest.mock.Mock(side_effect=AssertionError)}),
            self.assertRaisesRegex(ValidationFileError, "header seed"),
        ):
            self.authorize_fixture()

    def test_five_seed_loader_consumes_only_authenticated_bytes_and_exact_paths(self):
        real_runner = _run_validation_seed
        reader_name = "read_windows_artifact" if os.name == "nt" else "read_posix_artifact"
        reader = unittest.mock.Mock(wraps=globals()[reader_name])
        validation_reader = unittest.mock.Mock(wraps=read_predictions_file)

        def replace_after_authentication(payload, **kwargs):
            if kwargs["seed"] == 7:
                # Change the disk files after all authentication but before parsing predictions.
                for relative in (TRANSCRIPT_SOURCE, self.paths[7]):
                    (self.root / relative).write_bytes(b"changed after authentication")
            return real_runner(payload, **kwargs)

        with patch.dict(globals(), {reader_name: reader, "_run_validation_seed": replace_after_authentication,
                                    "read_predictions_file": validation_reader}):
            reports = self.authorize_fixture()
        self.assertEqual([call.args[0] for call in reader.call_args_list],
                         [self.root / relative for relative in self.payloads])
        self.assertEqual(tuple(reports), (7, 42, 99, 1337, 2023))
        self.assertIn("unclear", {state.value for state in CustomerState})
        self.assertEqual(validation_reader.call_count, 5)
        for call in validation_reader.call_args_list:
            self.assertEqual(call.args[1]["customer_state"], ["calm", "frustrated", "mad", "satisfied"])
            self.assertEqual(call.kwargs["expected_split"], "val")
        for index, (seed, report) in enumerate(reports.items()):
            self.assertEqual(report["scope"], "REAL VALIDATION")
            self.assertEqual(report["assignment_rate"], 1 / 1003)
            self.assertEqual(report["records"][0]["audio_file"], self.keys[index])
            self.assertEqual(report["records"][0]["transcript"], f"  Corrected text {index}!  ")
            self.assertEqual(report["provenance"]["file_path"], (self.root / self.paths[seed]).as_posix())
            self.assertEqual(report["provenance"]["sha256"], self.digests[self.paths[seed]])
            self.assertEqual(report["provenance"]["manifest_sha256"], self.digests[TRANSCRIPT_SOURCE])
            self.assertEqual(report["provenance"]["manifest_git_blob"], self.blob)

class PreparationTests(unittest.TestCase):
    def setUp(self):
        self.before = snapshot()
    def tearDown(self):
        self.assertEqual(snapshot(), self.before)
    def test_all_30_scores(self):
        for u, c, s in itertools.product(("high", "medium", "low"), ("at_risk", "not_at_risk"), ("mad", "frustrated", "calm", "satisfied", "unclear")):
            with self.subTest(urgency=u, churn=c, state=s):
                self.assertTrue(run_one(complaint_urgency=u, churn_risk=c, customer_state=s)["records"][0]["priority_verified"])
    def test_seven_missing_scoring_combinations(self):
        for count in (1, 2, 3):
            for missing in itertools.combinations(("complaint_urgency", "churn_risk", "customer_state"), count):
                report = run_one(**dict.fromkeys(missing))
                self.assertEqual(report["null_priority_count"], 1)
                self.assertEqual(report["human_review_count"], 1)
    def test_all_11_categories(self):
        self.assertEqual(len(ComplaintCategory), 11)
        for category in ComplaintCategory:
            report = run_one(complaint_category=category.value)
            self.assertEqual(report["assigned_count"], 1)
            self.assertEqual(report["skill_additions_approval"], "APPROVED_HAZEM")
    def test_missing_category(self):
        report = run_one(complaint_category=None)
        self.assertEqual(report["unassigned_reasons"], {"missing_category": 1})
        self.assertTrue(report["records"][0]["decision"]["route_reason"].strip())
    def test_versions(self):
        for field, bad in (("schema_version", 2), ("label_version", "old")):
            row, sup = fixture()
            sup[row["audio_file"]][field] = bad
            self.assertEqual(synthetic_batch([row], sup, META, capacity_mode="synthetic_sequential_release", synthetic=True)["invalid_rows"], 1)
            with self.assertRaises(ValueError):
                adapt_fixture(row, sup, dict(META, **{field: bad}))
    def test_off_shift(self):
        copies = [a.model_copy(deep=True) for a in agents.SEEDED_AGENTS]
        copies[0].on_shift = False
        with patch.object(agents, "SEEDED_AGENTS", copies):
            self.assertEqual(run_one()["unassigned_reasons"], {"off_shift": 1})
    def test_full_capacity(self):
        copies = [a.model_copy(deep=True) for a in agents.SEEDED_AGENTS]
        copies[0].current_load = copies[0].capacity
        with patch.object(agents, "SEEDED_AGENTS", copies):
            self.assertEqual(run_one()["unassigned_reasons"], {"capacity_exhausted": 1})
    def test_no_skill(self):
        with patch.object(agents, "SEEDED_AGENTS", []):
            self.assertEqual(run_one()["unassigned_reasons"], {"no_matching_skill": 1})
    def test_seed_order_and_tier_ignored(self):
        first = agents.SEEDED_AGENTS[0].model_copy(deep=True)
        second = first.model_copy(deep=True)
        second.id = "synthetic-second"
        first.tier, first.current_load = "L99", 4
        second.tier, second.current_load = "L1", 0
        with patch.object(agents, "SEEDED_AGENTS", [first, second]):
            self.assertEqual(run_one()["records"][0]["decision"]["assigned_agent_id"], first.id)
    def test_mixed_assigned_and_unassigned_metrics(self):
        assigned, supplements = fixture("synthetic-assigned")
        unassigned, extra = fixture("synthetic-unassigned", complaint_category=None)
        supplements.update(extra)
        report = synthetic_batch([assigned, unassigned], supplements, META,
                                 capacity_mode="synthetic_sequential_release", synthetic=True,
                                 manifest_rows=fixture_manifest([*supplements, "synthetic-m1", "synthetic-m2"]))
        self.assertEqual((report["assigned_count"], report["routing_unassigned_count"]), (1, 1))
        self.assertEqual(report["unassigned_count"], 3)
        self.assertEqual(report["assignment_rate"], 0.25)
        self.assertEqual(report["assignment_rate_denominator"], 4)
        self.assertEqual(report["input_coverage_rate"], 1.0)
        self.assertEqual(report["approved_validation_coverage_rate"], 0.5)

    def test_all_unassigned_metrics(self):
        with patch.object(agents, "SEEDED_AGENTS", []):
            report = run_one()
        self.assertEqual((report["assigned_count"], report["unassigned_count"]), (0, 1))
        self.assertEqual(report["assignment_rate"], 0.0)
        self.assertEqual(report["input_coverage_rate"], 1.0)

    def test_all_invalid_metrics(self):
        rows, supplements = [], {}
        for key in ("synthetic-invalid-a", "synthetic-invalid-b"):
            row, extra = fixture(key, complaint_category="unsupported")
            rows.append(row)
            supplements.update(extra)
        report = synthetic_batch(rows, supplements, META, capacity_mode="synthetic_sequential_release",
                                 synthetic=True)
        self.assertEqual(report["missing_approved_rows"], 0)
        self.assertEqual(report["invalid_approved_rows"], 2)
        self.assertEqual(len(report["contract_validation_errors"]), 2)
        self.assertEqual((report["valid_rows"], report["invalid_rows"], report["processed_rows"]), (0, 2, 0))
        self.assertEqual((report["assigned_count"], report["unassigned_count"]), (0, 2))
        self.assertEqual(report["assignment_rate"], 0.0)
        self.assertIsNone(report["assigned_over_processed_diagnostic"])
        self.assertEqual(report["assignment_rate_denominator"], 2)
        self.assertEqual(report["input_coverage_rate"], 0.0)
        self.assertEqual(report["approved_validation_coverage_rate"], 0.0)

    def test_seed_order_skips_ineligible_earlier_agents(self):
        for reason in ("skill", "shift", "capacity"):
            first = agents.SEEDED_AGENTS[0].model_copy(deep=True)
            second = first.model_copy(update={"id": "synthetic-later"}, deep=True)
            if reason == "skill":
                first.skills = []
            elif reason == "shift":
                first.on_shift = False
            else:
                first.current_load = first.capacity
            with self.subTest(reason=reason), patch.object(agents, "SEEDED_AGENTS", [first, second]):
                before = snapshot()
                report = run_one()
                self.assertEqual(report["records"][0]["decision"]["assigned_agent_id"], second.id)
                self.assertEqual(snapshot(), before)

    def test_ids_order_and_coverage(self):
        rows, sup = [], {}
        for key in ("synthetic-z", "synthetic-a", "synthetic-m"):
            row, extra = fixture(key); rows.append(row); sup.update(extra)
        report = synthetic_batch(rows, sup, META, capacity_mode="synthetic_sequential_release", synthetic=True)
        self.assertEqual([r["audio_file"] for r in report["records"]], [r["audio_file"] for r in rows])
        self.assertEqual([r["voicemail_id"] for r in report["records"]], [r["audio_file"] for r in rows])
        self.assertEqual(report["input_coverage_rate"], 1)
        self.assertEqual(report["approved_validation_coverage_rate"], 1.0)
    def test_duplicates_and_invalid_denominator(self):
        row, sup = fixture()
        valid, extra = fixture("synthetic-valid")
        sup.update(extra)
        report = synthetic_batch([row, row, {"bad": 1}, valid], sup, META,
                                 capacity_mode="synthetic_sequential_release", synthetic=True)
        self.assertEqual((report["processed_rows"], report["invalid_rows"]), (1, 3))
        self.assertEqual(report["assignment_rate"], 0.5)
        self.assertEqual(report["assignment_rate_denominator"], 2)
        self.assertEqual(report["input_coverage_rate"], 0.25)
        self.assertEqual(report["assigned_over_processed_diagnostic"], 1.0)
    def test_missing_fields_never_fabricated(self):
        row, sup = fixture()
        self.assertFalse(adapt_row(row, None, META, **fixture_inputs([row])).needs_human_review)
        for field in SUPPLEMENT_FIELDS:
            missing = copy.deepcopy(sup)
            missing[row["audio_file"]].pop(field)
            with self.subTest(field=field), self.assertRaises(ValueError):
                adapt_fixture(row, missing, META)
    def test_bad_labels_null_and_unsupported_rows(self):
        row, sup = fixture()
        for bad in ({"unexpected": 1}, dict(row, customer_state="invalid"), dict(row, confidence=1)):
            with self.assertRaises(ValueError):
                adapt_fixture(bad, sup, META)
        row["customer_state"] = None
        text = adapt_fixture(row, sup, META)
        self.assertIsNone(text.customer_state_text)
        self.assertTrue(text.needs_human_review)
        sup[row["audio_file"]].pop("confidence")
        with self.assertRaises(ValueError):
            adapt_fixture(row, sup, META)
    def test_sequential_capacity_reuse_and_exactly_once_release(self):
        first = agents.SEEDED_AGENTS[0].model_copy(deep=True)
        first.current_load = first.capacity - 1
        rows, supplements = [], {}
        for key in ("synthetic-one", "synthetic-two"):
            row, extra = fixture(key)
            rows.append(row)
            supplements.update(extra)
        with patch.object(agents, "SEEDED_AGENTS", [first]), patch.object(agents, "release", wraps=agents.release) as release:
            before = snapshot()
            report = synthetic_batch(rows, supplements, META, capacity_mode="synthetic_sequential_release", synthetic=True)
            self.assertEqual([r["decision"]["assigned_agent_id"] for r in report["records"]], [first.id, first.id])
            self.assertEqual([call.args for call in release.call_args_list], [(first.id,), (first.id,)])
            self.assertEqual(snapshot(), before)
            self.assertEqual(report["metrics_scope"], "sequential_validation")

    def test_cleanup_on_verification_failure(self):
        with patch.dict(globals(), {"priority_expected": lambda text: (-1, "low")}), \
                patch.object(agents, "release", wraps=agents.release) as release:
            with self.assertRaises(AssertionError):
                run_one()
            release.assert_called_once()
            self.assertEqual(snapshot(), self.before)
    def test_official_triage_rolls_back_failed_decision_once(self):
        from svr.stages import triage
        with (
            patch.object(triage, "TriageDecision", side_effect=RuntimeError("synthetic construction failure")),
            patch.object(agents, "release", wraps=agents.release) as release,
        ):
            with self.assertRaisesRegex(RuntimeError, "synthetic construction failure"):
                run_one()
            release.assert_called_once()
            self.assertEqual(snapshot(), self.before)

    def test_official_execution_and_loading_disabled(self):
        with self.assertRaises(PermissionError):
            synthetic_batch([], {}, META, capacity_mode="synthetic_sequential_release")
        with self.assertRaises(ValueError):
            synthetic_batch([], {}, META, capacity_mode="unspecified", synthetic=True)
        with self.assertRaises(PermissionError):
            load_validation_predictions("never-opened", approvals={"all": True})
        with self.assertRaises(ValueError):
            validate_meta(dict(META, split="test"))
    def test_voice_fusion_disabled(self):
        row, sup = fixture()
        text = adapt_fixture(row, sup, META)
        decision = None
        try:
            voice = VoiceEvidence(voicemail_id=text.voicemail_id, model_version="synthetic-voice",
                                  voice_state_candidate="mad", voice_confidence=1)
            decision = triage_phase1(text, voice)
            score, bucket = priority_expected(text)
            self.assertEqual(decision.priority_score, score)
            self.assertEqual(decision.queue_priority.value, bucket)
            self.assertIn("voice_evidence=ignored_fusion_disabled", decision.contributing_signals)
        finally:
            if decision and decision.assigned_agent_id:
                agents.release(decision.assigned_agent_id)
    def test_corrected_text_join_and_input_order(self):
        first, supplements = fixture("synthetic-b")
        second, extra = fixture("synthetic-a")
        supplements.update(extra)
        rows = [first, second]
        inputs = fixture_inputs(rows)
        corrected = "  " + "x" * 205 + "\n!"
        inputs["transcripts"] = index_transcripts([
            {"audio_file": "synthetic-a", "text": "Second!", "text_norm": "second"},
            {"audio_file": "synthetic-b", "text": corrected, "text_whisper": "Wrong transcript"},
        ])
        report = run_batch(rows, supplements, META, capacity_mode="synthetic_sequential_release",
                           synthetic=True, manifest_rows=fixture_manifest(supplements), **inputs)
        self.assertEqual([r["transcript"] for r in report["records"]], [corrected, "Second!"])
        text = adapt_row(first, supplements, META, **inputs)
        self.assertEqual(text.summary, corrected[:200])

    def test_summary_python_character_boundaries(self):
        for length in (1, 199, 200, 201):
            corrected = "\U0001F642" * length
            with self.subTest(length=length):
                self.assertEqual(summarize_transcript(corrected), "\U0001F642" * min(length, 200))

    def test_missing_and_invalid_corrected_text(self):
        row, supplements = fixture()
        for bad in (None, "", " \t", 42):
            with self.subTest(text=bad), self.assertRaises(ValueError):
                summarize_transcript(bad)
        inputs = fixture_inputs([row])
        inputs["transcripts"] = index_transcripts([
            {"audio_file": row["audio_file"], "text_norm": "fallback", "text_whisper": "fallback"}
        ])
        with self.assertRaisesRegex(ValueError, "Corrected text"):
            adapt_row(row, supplements, META, **inputs)
        inputs["transcripts"] = {}
        with self.assertRaisesRegex(ValueError, "Missing corrected text"):
            adapt_row(row, supplements, META, **inputs)
        with self.assertRaises(ValueError):
            index_transcripts([{"audio_file": "duplicate"}] * 2)

    def test_customer_request_categories(self):
        self.assertEqual(set(CATEGORY_REQUESTS), {category.value for category in ComplaintCategory})
        self.assertEqual(len(set(CATEGORY_REQUESTS.values())), 11)
        for category in ComplaintCategory:
            with self.subTest(category=category):
                row, supplements = fixture(complaint_category=category.value)
                text = adapt_fixture(row, supplements, META)
                self.assertEqual(text.customer_request, category_customer_request(category))
                self.assertTrue(text.customer_request.endswith("."))
        self.assertEqual(category_customer_request(None), "")
        row, supplements = fixture(complaint_category=None)
        self.assertEqual(adapt_fixture(row, supplements, META).customer_request, "")
        for bad in ("unsupported", "", 42):
            with self.subTest(category=bad), self.assertRaises(ValueError):
                category_customer_request(bad)

    def test_confidence_threshold_boundaries_without_upstream_review(self):
        row, supplements = fixture()
        inputs = fixture_inputs([row])
        scores = inputs["probabilities"][row["audio_file"]]
        for field in LABEL_MAP:
            with self.subTest(field=field):
                scores[field][row[field]] = 0.49
                scores[field]["different_class"] = 0.01
                text = adapt_row(row, supplements, META, **inputs)
                self.assertEqual(text.confidence, 0.49)
                self.assertTrue(text.needs_human_review)
                scores[field][row[field]] = 0.9
        scores["churn_risk"][row["churn_risk"]] = 0.5
        self.assertFalse(adapt_row(row, supplements, META, **inputs).needs_human_review)
        self.assertTrue(adapt_row(row, supplements, META, review_threshold=0.51, **inputs).needs_human_review)
        self.assertFalse(adapt_row(row, None, META, review_threshold=0.49, **inputs).needs_human_review)
        inputs["probabilities"] = {}
        with self.assertRaisesRegex(ValueError, "Missing predicted-class probability"):
            adapt_row(row, supplements, META, **inputs)

    def test_unassigned_triggers_final_review(self):
        with patch.object(agents, "SEEDED_AGENTS", []):
            report = run_one()
            self.assertEqual(report["human_review_count"], 1)
            self.assertEqual(report["unassigned_count"], 1)

    def test_proposed_scripts_match_all_categories_and_feed_adapter(self):
        self.assertEqual(set(PROPOSED_SUGGESTED_SCRIPTS), set(ComplaintCategory))
        self.assertEqual(len(set(PROPOSED_SUGGESTED_SCRIPTS.values())), 11)
        for category in ComplaintCategory:
            with self.subTest(category=category):
                draft = PROPOSED_SUGGESTED_SCRIPTS[category]
                self.assertTrue(draft.strip())
                self.assertEqual(suggest_script(category), draft)
                self.assertEqual(suggest_script(category.value), draft)
                row, supplements = fixture(complaint_category=category.value)
                supplied = supplements[row["audio_file"]]
                supplied.pop("suggested_script")
                self.assertEqual(adapt_fixture(row, supplements, META).suggested_script, draft)
                supplied["suggested_script"] = "Unrelated upstream script"
                self.assertEqual(adapt_fixture(row, supplements, META).suggested_script, draft)

    def test_script_lookup_rejects_unsupported_categories(self):
        for category in (None, "", "NETWORK_AND_CONNECTIVITY", "unknown", 42):
            with self.subTest(category=category), self.assertRaises(ValueError):
                suggest_script(category)

    def test_null_category_requires_explicit_nonblank_script(self):
        row, supplements = fixture(complaint_category=None)
        supplied = supplements[row["audio_file"]]
        supplied["suggested_script"] = "Explicit upstream draft"
        self.assertEqual(adapt_fixture(row, supplements, META).suggested_script, "Explicit upstream draft")
        for bad in (None, "", "   ", 42):
            with self.subTest(script=bad), self.assertRaisesRegex(ValueError, "explicit upstream script"):
                supplied["suggested_script"] = bad
                adapt_fixture(row, supplements, META)
        supplied.pop("suggested_script")
        with self.assertRaisesRegex(ValueError, "explicit upstream script"):
            adapt_fixture(row, supplements, META)

    def test_batch_requires_keyed_probability_map(self):
        row, supplements = fixture()
        transcripts = fixture_inputs([row])["transcripts"]
        options = dict(capacity_mode="synthetic_sequential_release", synthetic=True, transcripts=transcripts)
        with self.assertRaisesRegex(ValueError, "probability maps"):
            run_batch([row], supplements, META, **options)
        for bad in (None, []):
            with self.subTest(probabilities=bad), self.assertRaisesRegex(ValueError, "probability maps"):
                run_batch([row], supplements, META, probabilities=bad, **options)

    def test_missing_row_probability_is_indexed_and_other_rows_continue(self):
        first, supplements = fixture("synthetic-missing")
        second, extra = fixture("synthetic-valid")
        supplements.update(extra)
        inputs = fixture_inputs([first, second])
        del inputs["probabilities"][first["audio_file"]]
        report = run_batch([first, second], supplements, META, **inputs, manifest_rows=fixture_manifest(supplements),
                           capacity_mode="synthetic_sequential_release", synthetic=True)
        self.assertEqual([r["row_index"] for r in report["contract_validation_errors"]], [0])
        self.assertIn("Missing predicted-class probability", report["contract_validation_errors"][0]["error"])
        self.assertEqual([r["row_index"] for r in report["records"]], [1])

    def test_invalid_numeric_confidence_and_threshold_inputs(self):
        row, supplements = fixture()
        inputs = fixture_inputs([row])
        scores = inputs["probabilities"][row["audio_file"]]["churn_risk"]
        for bad in (True, "0.5", float("nan"), float("inf"), -0.01, 1.01):
            with self.subTest(value=bad):
                with self.assertRaisesRegex(ValueError, "review_threshold"):
                    run_batch([row], supplements, META, **inputs, review_threshold=bad,
                              capacity_mode="synthetic_sequential_release", synthetic=True)
                scores[row["churn_risk"]] = bad
                with self.assertRaisesRegex(ValueError, "predicted-class probability"):
                    adapt_row(row, supplements, META, **inputs)

    def test_batch_review_threshold_preserves_priority_routing_and_draft(self):
        row, supplements = fixture()
        inputs = fixture_inputs([row])
        options = dict(capacity_mode="synthetic_sequential_release", synthetic=True,
                       manifest_rows=fixture_manifest(supplements), **inputs)
        normal = run_batch([row], supplements, META, **options)
        with patch.dict(globals(), {"triage_phase1": unittest.mock.Mock(wraps=triage_phase1)}) as patched:
            review = run_batch([row], supplements, META, review_threshold=1, **options)
            text = patched["triage_phase1"].call_args.args[0]
        self.assertTrue(text.needs_human_review)
        self.assertEqual(text.suggested_script, suggest_script(row["complaint_category"]))
        self.assertEqual((normal["human_review_count"], review["human_review_count"]), (0, 1))
        for field in ("assigned_agent_id", "priority_score", "queue_priority"):
            self.assertEqual(normal["records"][0]["decision"][field], review["records"][0]["decision"][field])

    def test_fixed_1003_manifest_denominator_counts_invalid_and_missing(self):
        assigned, supplements = fixture("synthetic-assigned")
        unrouted, extra = fixture("synthetic-unrouted", complaint_category=None)
        supplements.update(extra)
        invalid, extra = fixture("synthetic-invalid", complaint_category="unsupported")
        supplements.update(extra)
        manifest = fixture_manifest([*supplements, *(
            f"synthetic-missing-{i}" for i in range(APPROVED_VALIDATION_ROW_COUNT - len(supplements))
        )])
        report = synthetic_batch([invalid, assigned, unrouted], supplements, META,
                                 manifest_rows=manifest, capacity_mode="synthetic_sequential_release", synthetic=True)
        self.assertEqual(report["assignment_rate_denominator"], 1003)
        self.assertEqual(report["assignment_rate"], 1 / 1003)
        self.assertEqual(report["approved_validation_coverage_rate"], 2 / 1003)
        self.assertEqual(report["unassigned_count"], 1002)
        self.assertEqual(report["routing_unassigned_count"], 1)
        self.assertEqual(report["invalid_approved_rows"], 1)
        self.assertEqual(report["missing_approved_rows"], 1000)
        self.assertEqual(report["invalid_rows"], 1)
        self.assertIn("unsupported", report["contract_validation_errors"][0]["error"])
        self.assertEqual([record["row_index"] for record in report["records"]], [1, 2])
        self.assertEqual(len(report["missing_predictions"]), 1000)

    def test_manifest_reconciliation_rejects_all_duplicate_occurrences(self):
        duplicate, supplements = fixture("synthetic-duplicate")
        valid, extra = fixture("synthetic-valid")
        supplements.update(extra)
        unexpected, extra = fixture("synthetic-unexpected")
        supplements.update(extra)
        manifest = fixture_manifest(["synthetic-duplicate", "synthetic-valid", "synthetic-missing"])
        report = synthetic_batch([duplicate, unexpected, valid, duplicate], supplements, META,
                                 manifest_rows=manifest, capacity_mode="synthetic_sequential_release", synthetic=True)
        self.assertEqual(report["manifest_reconciliation"], {
            "missing_keys": [("val", "synthetic-missing")],
            "duplicate_keys": [("val", "synthetic-duplicate")],
            "unexpected_keys": [("val", "synthetic-unexpected")],
        })
        self.assertEqual(report["assignment_rate"], 1 / 3)
        self.assertEqual(report["invalid_approved_rows"], 1)
        self.assertEqual(report["invalid_rows"], 3)
        self.assertEqual([(e["row_index"], e["error"]) for e in report["contract_validation_errors"]], [
            (0, "Duplicate prediction composite key"),
            (1, "Unexpected prediction composite key"),
            (3, "Duplicate prediction composite key"),
        ])
        self.assertEqual([record["row_index"] for record in report["records"]], [2])

    def test_manifest_rejects_duplicates_wrong_split_and_missing_keys(self):
        entry = {"split": "val", "audio_file": "synthetic-a"}
        for rows, reason in (
            ([entry, entry], "Duplicate approved manifest key"),
            ([dict(entry, split="train")], "must have split val"),
            ([{"split": "val"}], "must have a nonempty audio_file"),
        ):
            with self.subTest(reason=reason), self.assertRaisesRegex(ValueError, reason):
                validation_manifest_keys(rows)

    def test_metadata_preserved(self):
        self.assertEqual(run_one()["metadata"], META)
        self.assertIsNone(run_one()["official_validation_result"])
    def test_empty_batch(self):
        for manifest, expected_coverage in (([], None), (fixture_manifest(["synthetic-absent"]), 0.0)):
            with self.subTest(manifest=manifest):
                report = synthetic_batch([], {}, META, capacity_mode="synthetic_sequential_release",
                                         synthetic=True, manifest_rows=manifest)
                self.assertEqual((report["total_input_rows"], report["processed_rows"]), (0, 0))
                self.assertEqual(report["assignment_rate"], expected_coverage)
                self.assertIsNone(report["assigned_over_processed_diagnostic"])
                self.assertIsNone(report["input_coverage_rate"])
                self.assertEqual(report["approved_validation_coverage_rate"], expected_coverage)

    def test_no_files_opened_by_batch(self):
        with patch("builtins.open", side_effect=AssertionError("No file access")), patch.object(Path, "open", side_effect=AssertionError("No path access")):
            self.assertEqual(run_one()["processed_rows"], 1)
            with self.assertRaises(PermissionError):
                load_validation_predictions("forbidden-file")
    def test_duplicate_voicemail_ids(self):
        first, sup = fixture("synthetic-first")
        second, extra = fixture("synthetic-second")
        extra[second["audio_file"]]["voicemail_id"] = sup[first["audio_file"]]["voicemail_id"]
        sup.update(extra)
        report = synthetic_batch([first, second], sup, META, capacity_mode="synthetic_sequential_release", synthetic=True)
        self.assertEqual((report["processed_rows"], report["invalid_rows"]), (0, 2))
    def test_official_assembler_handoff(self):
        from svr.schemas import AudioEvidence
        for category in ("network_and_connectivity", None):
            row, sup = fixture(complaint_category=category)
            text = adapt_fixture(row, sup, META)
            decision = None
            try:
                decision = triage_phase1(text)
                corrected = run_one(complaint_category=category)["records"][0]["transcript"]
                audio = AudioEvidence(voicemail_id=text.voicemail_id, format="wav",
                    transcript=corrected, stt_provider="synthetic", stt_model_version="synthetic-v1")
                ticket = assemble_ticket(ticket_id="synthetic-ticket", voicemail_id=text.voicemail_id,
                    audio=audio, text=text, voice_evidence=None, decision=decision,
                    stage_status={"triage": "ok"}, stage_timing_ms={"triage": 0.0}, processing_ms=0.0)
                self.assertEqual(ticket.transcript, corrected)
                self.assertIsNone(ticket.customer_state_final)
                self.assertEqual(ticket.needs_human_review, category is None)
                self.assertEqual(ticket.ticket_status.value, "needs_review" if category is None else "completed")
            finally:
                if decision and decision.assigned_agent_id:
                    agents.release(decision.assigned_agent_id)

stream = io.StringIO()
result = unittest.TextTestRunner(stream=stream, verbosity=2).run(unittest.TestSuite(
    unittest.defaultTestLoader.loadTestsFromTestCase(case) for case in (PreparationTests, PredictionHandoffTests, ApprovedValidationSeedTests, GatedValidationLoaderTests)
))
print(stream.getvalue())
TEST_COUNTS = {"passed": result.testsRun - len(result.failures) - len(result.errors) - len(result.skipped),
               "failed": len(result.failures), "error": len(result.errors), "skipped": len(result.skipped)}
print(json.dumps(TEST_COUNTS))
assert result.wasSuccessful()
assert snapshot() == BASELINE

test_all_11_categories (__main__.PreparationTests.test_all_11_categories) ... ok
test_all_30_scores (__main__.PreparationTests.test_all_30_scores) ... ok
test_all_invalid_metrics (__main__.PreparationTests.test_all_invalid_metrics) ... ok
test_all_unassigned_metrics (__main__.PreparationTests.test_all_unassigned_metrics) ... ok
test_bad_labels_null_and_unsupported_rows (__main__.PreparationTests.test_bad_labels_null_and_unsupported_rows) ... ok
test_batch_requires_keyed_probability_map (__main__.PreparationTests.test_batch_requires_keyed_probability_map) ... ok
test_batch_review_threshold_preserves_priority_routing_and_draft (__main__.PreparationTests.test_batch_review_threshold_preserves_priority_routing_and_draft) ... ok
test_cleanup_on_verification_failure (__main__.PreparationTests.test_cleanup_on_verification_failure) ... ok
test_confidence_threshold_boundaries_without_upstream_review (__main__.PreparationTests.test_confidence_threshold_boundaries_without_upstream_review) ... ok
t

## 15. Pending dependencies and limitations
Before an authorized real run, I verify the ready-branch val manifest has exactly 1,003 unique keys and reconcile the fair-run MARBERTv2 artifacts against the corrected transcripts. I exercise the gated loader with synthetic fixtures only. A real run requires separate authorization and the loader authenticates the exact approved files before routing. Hazem's customer-facing script review is pending.

The current reader rejects null prediction labels. The adapter validates nulls at a proposed future handoff boundary without claiming that the existing file format supports them. Classification metrics are not routing metrics. Real file loading and official execution require explicit authorization. The saved outputs contain synthetic results only.

All 11 suggested scripts are proposed drafts. I preserve the supplied wording for nine and generalize the branch/staff and waiting drafts for Hazem's review.